In [2]:
#!/usr/bin/env python3
"""
RSA Lag Analysis — Hippocampus Only (LHP / RHP), Band × Pair (Long Format)
---------------------------------------------------------------------------
Runs the RSA lag pipeline for:
  Regions : LHP, RHP  (hippocampus only)
  Bands   : theta  (3–7 Hz,   freq idx 0–3)
            alpha  (7–12 Hz,  freq idx 4–6)
            beta   (12–40 Hz, freq idx 7–11)
            gamma  (40–128 Hz,freq idx 12–17)

For every trial, for every pair of clean recalls (i, j) with i < j,
for every band:
  - Slice the 18-freq oscillatory IRASA vector to the band's freq indices
    → gives a (N_channels × N_band_freqs) sub-matrix per recall
  - Flatten each sub-matrix to a 1-D vector
  - RSA_r_ret = Pearson r between retrieval vectors (output positions i, j)
  - RSA_r_enc = Pearson r between encoding vectors  (serial positions of i, j)

Output is LONG format: one row per pair × band.

Output columns:
  subject, session, experiment, trial,
  region, hemisphere,
  band, band_freq_indices,
  output_pos_i, output_pos_j, output_lag,
  word_i, word_j,
  serial_pos_i, serial_pos_j, T_lag, SP_lag,
  RSA_r_ret, RSA_r_enc, n_channels, semantic_sim

Per-region × per-band CSVs:
  ./rsa_lag_hc_bands/ALL_SUBJECTS_{exp}_{region}_{band}_rsa_lag.csv

Per-region all-bands CSV:
  ./rsa_lag_hc_bands/ALL_SUBJECTS_{exp}_{region}_allbands_rsa_lag.csv

Combined (both HC regions × all bands):
  ./rsa_lag_hc_bands/ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag.csv
"""

import warnings
import traceback
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS = ['DBOY1', 'EFRCourierReadOnly', 'EFRCourierOpenLoop']

INPUT_DIRS = {
    'DBOY1':               Path('./subject_results_DBOY1'),
    'EFRCourierReadOnly':  Path('./subject_results_EFRCourierReadOnly'),
    'EFRCourierOpenLoop':  Path('./subject_results_EFRCourierOpenLoop'),
}

OUTPUT_DIR = Path('./rsa_lag_hc_bands_frac')
OUTPUT_DIR.mkdir(exist_ok=True)

# Hippocampus only
REGIONS    = ['LHP', 'RHP']
HEMISPHERE = {'LHP': 'left', 'RHP': 'right'}

# 18 log-spaced frequencies from 3–128 Hz:
# idx:  0     1     2     3     4     5      6      7      8
# Hz:  3.00  3.74  4.67  5.82  7.26  9.05  11.28  14.07  17.55
# idx:  9     10    11    12    13    14    15     16     17
# Hz: 21.88 27.29 34.03 42.44 52.92 66.00 82.31 102.64 128.00
N_FREQS = 18

BANDS = {
    'theta': list(range(1, 5)),   # 3.00 – 5.82 Hz
    'alpha': list(range(5, 7)),   # 7.26 – 11.28 Hz
    'beta':  list(range(7, 12)),  # 14.07 – 34.03 Hz
    'gamma': list(range(12, 18)), # 42.44 – 128.00 Hz
}
BAND_ORDER = ['theta', 'alpha', 'beta', 'gamma']

RET_OSC_COLS = [f'ret_frac_f{i:02d}' for i in range(N_FREQS)]
ENC_OSC_COLS = [f'enc_frac_f{i:02d}' for i in range(N_FREQS)]

# ---- Word2Vec ---------------------------------------------------------------
WORD2VEC_PATH = Path('/home1/noaherz/word2vec/GoogleNews-vectors-negative300.bin.gz')

OUTPUT_COLS = [
    'subject', 'session', 'experiment', 'trial',
    'region', 'hemisphere',
    'band', 'band_freq_indices',
    'output_pos_i', 'output_pos_j', 'output_lag',
    'word_i', 'word_j',
    'serial_pos_i', 'serial_pos_j', 'T_lag', 'SP_lag',
    'RSA_r_ret', 'RSA_r_enc',   # <-- split from original RSA_r
    'n_channels', 'semantic_sim',
]

# ============================================================================
# WORD2VEC LOADER + SIMILARITY
# ============================================================================

def load_word2vec(path: Path):
    if path is None or not path.exists():
        print(f"  ⚠  Word2Vec model not found at {path}. semantic_sim will be NaN.")
        return None
    try:
        import gensim.models as gensim_models
        print(f"  Loading Word2Vec from {path} …")
        model = gensim_models.KeyedVectors.load_word2vec_format(str(path), binary=True)
        try:
            vs = len(model)
        except TypeError:
            vs = len(model.vocab)
        print(f"  ✓ Word2Vec loaded — vocab: {vs:,}")
        return model
    except Exception as e:
        print(f"  ✗ Word2Vec load failed: {e}. semantic_sim will be NaN.")
        return None


def case_insensitive_similarity(word1: str, word2: str, model) -> Optional[float]:
    cases = [
        (word1.lower(), word2.lower()),
        (word1.lower(), word2.upper()),
        (word1.upper(), word2.lower()),
        (word1.upper(), word2.upper()),
    ]
    sims = []
    for w1, w2 in cases:
        try:
            sims.append(model.similarity(w1, w2))
        except KeyError:
            continue
    return float(max(sims)) if sims else None


def build_similarity_cache(words: set, model) -> dict:
    if model is None:
        return {}
    unique_words = sorted(w for w in words if isinstance(w, str))
    n = len(unique_words)
    print(f"    Building semsim cache: {n} words → {n*(n-1)//2} pairs …")
    cache = {}
    for i, w1 in enumerate(unique_words):
        for w2 in unique_words[i:]:
            key = frozenset({w1, w2})
            if key not in cache:
                sim = case_insensitive_similarity(w1, w2, model)
                cache[key] = sim if sim is not None else np.nan
    return cache


# ============================================================================
# VECTOR + STATISTICS HELPERS
# ============================================================================

def build_band_vector(recall_rows: pd.DataFrame,
                      channel_index,
                      band_freq_indices: list,
                      phase_cols: list) -> np.ndarray:
    """
    Build a 1-D vector for one event and one band:
      1. Align rows to channel_index       → (N_ch, 18) matrix
      2. Slice to band_freq_indices        → (N_ch, N_band_freqs) sub-matrix
      3. Flatten                           → (N_ch * N_band_freqs,) vector

    phase_cols : either RET_OSC_COLS or ENC_OSC_COLS (18 elements)
    Missing channels are NaN (handled pairwise in safe_pearsonr).
    """
    ch_df = (
        recall_rows
        .drop_duplicates(subset='channel_idx')
        .set_index('channel_idx')
        .reindex(channel_index)
    )
    band_cols = [phase_cols[i] for i in band_freq_indices]
    mat = ch_df[band_cols].values   # (N_ch, N_band_freqs)
    return mat.flatten()


def safe_pearsonr(v1: np.ndarray, v2: np.ndarray) -> float:
    if len(v1) != len(v2):
        return np.nan
    mask = np.isfinite(v1) & np.isfinite(v2)
    if mask.sum() < 3:
        return np.nan
    r, _ = pearsonr(v1[mask], v2[mask])
    return float(r)


def safe_euclidean(x1, z1, x2, z2) -> float:
    if any(not np.isfinite(v) for v in (x1, z1, x2, z2)):
        return np.nan
    return float(euclidean([x1, z1], [x2, z2]))


def extract_scalar(series: pd.Series, field: str, context: str):
    unique_vals = series.dropna().unique()
    if len(unique_vals) > 1:
        warnings.warn(
            f"[{context}] '{field}' has {len(unique_vals)} distinct values "
            f"({unique_vals[:3]}…). Taking first.")
    return series.iloc[0]


# ============================================================================
# TRIAL-LEVEL PROCESSING
# ============================================================================

def process_trial_region(trial_df: pd.DataFrame,
                         region: str,
                         sim_cache: dict) -> List[Dict]:
    """
    For one (subject, session, trial, region):
      - Enumerate all valid recall pairs (i, j), i < j
      - For each pair × each band: compute RSA_r_ret and RSA_r_enc
        RSA_r_ret: Pearson r on retrieval osc vectors (at output positions i, j)
        RSA_r_enc: Pearson r on encoding  osc vectors (at serial positions of i, j)
      - Returns list of dicts in OUTPUT_COLS order (long format)

    Encoding vectors are indexed by serial_position (which word was presented),
    NOT by recall output position — so we look up enc_osc for the channel rows
    where recall_output_position == op_i/op_j, which already carry the correct
    enc_osc values for that word's encoding epoch (set in the channel-wide pipeline).
    """
    rows = []

    output_positions = sorted(
        trial_df['recall_output_position'].unique(),
        key=lambda x: int(x),
    )
    if len(output_positions) < 2:
        return rows

    channel_index = sorted(trial_df['channel_idx'].unique(), key=int)
    sample_row    = trial_df.iloc[0]

    # ------------------------------------------------------------------
    # Pre-compute per-output-position metadata + per-band vectors
    # (both retrieval and encoding)
    # ------------------------------------------------------------------
    pos_data: Dict[int, Dict] = {}

    for op in output_positions:
        op_rows = trial_df[trial_df['recall_output_position'] == op]
        ctx = (f"subj={sample_row['subject']} sess={sample_row['session']} "
               f"trial={sample_row['trial']} region={region} op={op}")

        word       = extract_scalar(op_rows['recalled_word'],   'recalled_word',   ctx)
        serial_pos = extract_scalar(op_rows['serial_position'], 'serial_position', ctx)
        store_x    = extract_scalar(op_rows['store_x'],         'store_x',         ctx)
        store_z    = extract_scalar(op_rows['store_z'],         'store_z',         ctx)
        n_ch       = op_rows['channel_idx'].nunique()

        # Retrieval vectors: one per band, sliced from ret_osc cols
        ret_band_vectors = {
            band_name: build_band_vector(op_rows, channel_index, freq_idx, RET_OSC_COLS)
            for band_name, freq_idx in BANDS.items()
        }

        # Encoding vectors: one per band, sliced from enc_osc cols
        # The enc_osc values on these rows correspond to the encoding epoch
        # of the recalled word (serial_pos), written there by the channel-wide
        # pipeline — so we can read them directly from op_rows.
        enc_band_vectors = {
            band_name: build_band_vector(op_rows, channel_index, freq_idx, ENC_OSC_COLS)
            for band_name, freq_idx in BANDS.items()
        }

        pos_data[op] = {
            'ret_band_vectors': ret_band_vectors,
            'enc_band_vectors': enc_band_vectors,
            'word':             word,
            'serial_pos':       serial_pos,
            'store_x':          store_x,
            'store_z':          store_z,
            'n_channels':       n_ch,
        }

    subject    = sample_row['subject']
    session    = sample_row['session']
    experiment = sample_row['experiment']
    trial      = sample_row['trial']

    # ------------------------------------------------------------------
    # Enumerate pairs × bands
    # ------------------------------------------------------------------
    for idx_i, op_i in enumerate(output_positions):
        for op_j in output_positions[idx_i + 1:]:
            d_i = pos_data[op_i]
            d_j = pos_data[op_j]

            output_lag = int(op_j) - int(op_i)
            T_lag      = abs(int(d_i['serial_pos']) - int(d_j['serial_pos']))
            SP_lag     = safe_euclidean(d_i['store_x'], d_i['store_z'],
                                        d_j['store_x'], d_j['store_z'])
            n_channels = min(d_i['n_channels'], d_j['n_channels'])

            # Semantic similarity: same across all bands — compute once
            w_i = str(d_i['word']).lower() if pd.notna(d_i['word']) else None
            w_j = str(d_j['word']).lower() if pd.notna(d_j['word']) else None
            sem_sim = (
                sim_cache.get(frozenset({w_i, w_j}), np.nan)
                if (w_i and w_j and sim_cache)
                else np.nan
            )

            # One row per band
            for band_name in BAND_ORDER:
                freq_idx = BANDS[band_name]

                # Retrieval RSA
                v_ret_i  = d_i['ret_band_vectors'][band_name]
                v_ret_j  = d_j['ret_band_vectors'][band_name]
                rsa_ret  = safe_pearsonr(v_ret_i, v_ret_j)

                # Encoding RSA
                v_enc_i  = d_i['enc_band_vectors'][band_name]
                v_enc_j  = d_j['enc_band_vectors'][band_name]
                rsa_enc  = safe_pearsonr(v_enc_i, v_enc_j)

                rows.append({
                    'subject':           subject,
                    'session':           session,
                    'experiment':        experiment,
                    'trial':             trial,
                    'region':            region,
                    'hemisphere':        HEMISPHERE[region],
                    'band':              band_name,
                    'band_freq_indices': str(freq_idx),
                    'output_pos_i':      op_i,
                    'output_pos_j':      op_j,
                    'output_lag':        output_lag,
                    'word_i':            d_i['word'],
                    'word_j':            d_j['word'],
                    'serial_pos_i':      d_i['serial_pos'],
                    'serial_pos_j':      d_j['serial_pos'],
                    'T_lag':             T_lag,
                    'SP_lag':            SP_lag,
                    'RSA_r_ret':         rsa_ret,
                    'RSA_r_enc':         rsa_enc,
                    'n_channels':        n_channels,
                    'semantic_sim':      sem_sim,
                })

    return rows


# ============================================================================
# PER-EXPERIMENT RUNNER
# ============================================================================

def run_experiment(exp: str, w2v_model):
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {exp}")
    print(f"{'='*70}")

    input_dir  = INPUT_DIRS.get(exp)
    if input_dir is None:
        print(f"  ✗ No INPUT_DIR configured for '{exp}'.")
        return

    input_path = input_dir / f"ALL_SUBJECTS_{exp}_irasa_channel_wide.csv"
    if not input_path.exists():
        print(f"  ✗ Master CSV not found: {input_path}")
        return

    print(f"  Loading {input_path} …")
    df = pd.read_csv(input_path)
    print(f"  Loaded {len(df):,} rows | "
          f"{df['subject'].nunique()} subjects | "
          f"{df['session'].nunique()} sessions")

    # Filter to HC only upfront
    df = df[df['region'].isin(REGIONS)].copy()
    print(f"  After HC filter: {len(df):,} rows")
    if df.empty:
        print("  ✗ No hippocampal data found — check 'region' column values.")
        return

    df['channel_idx'] = df['channel_idx'].astype(int)

    # Build Word2Vec cache once per experiment (all HC words)
    all_words_lower = set(
        df['recalled_word'].dropna().astype(str).str.lower().unique()
    )
    sim_cache = build_similarity_cache(all_words_lower, w2v_model)

    all_region_dfs = []

    for region in REGIONS:
        print(f"\n  {'─'*60}")
        print(f"  Region: {region}")

        region_df = df[df['region'] == region].copy()
        if region_df.empty:
            print(f"  ✗ No rows for region {region} — skipping")
            continue

        print(f"  Rows: {len(region_df):,}")

        all_rows = []
        groups   = region_df.groupby(['subject', 'session', 'trial'])
        n_groups = len(groups)

        for g_idx, ((subj, sess, trial), trial_df) in enumerate(groups):
            if g_idx % 200 == 0:
                print(f"    Processing trial group {g_idx}/{n_groups} …")
            try:
                rows = process_trial_region(trial_df, region, sim_cache)
                all_rows.extend(rows)
            except Exception as e:
                print(f"    FAILED [{subj} sess={sess} trial={trial}]: {e}")
                traceback.print_exc()
                continue

        if not all_rows:
            print(f"  ✗ No pairs generated for region {region}")
            continue

        result_df = pd.DataFrame(all_rows, columns=OUTPUT_COLS)
        all_region_dfs.append(result_df)

        # Per-region × per-band outputs
        for band_name in BAND_ORDER:
            band_df  = result_df[result_df['band'] == band_name]
            out_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_{region}_{band_name}_rsa_lag.csv"
            band_df.to_csv(out_path, index=False)
            print(f"    ✓ {region}/{band_name}: {len(band_df):,} pairs | "
                  f"RSA_ret NaN={band_df['RSA_r_ret'].isna().mean()*100:.1f}% | "
                  f"RSA_enc NaN={band_df['RSA_r_enc'].isna().mean()*100:.1f}% | "
                  f"SemSim NaN={band_df['semantic_sim'].isna().mean()*100:.1f}% | "
                  f"→ {out_path.name}")

        # All-bands file for this region
        reg_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_{region}_allbands_rsa_lag.csv"
        result_df.to_csv(reg_path, index=False)
        print(f"  ✓ {region} all-bands → {reg_path.name}")

    # Combined: both HC regions × all bands
    if all_region_dfs:
        combined  = pd.concat(all_region_dfs, ignore_index=True)
        comb_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag.csv"
        combined.to_csv(comb_path, index=False)
        print(f"\n  ✓ Combined HC all-bands → {comb_path.name}")
        print(f"    Total rows : {len(combined):,}")
        print(f"    Regions    : {combined['region'].unique().tolist()}")
        print(f"    Bands      : {combined['band'].unique().tolist()}")
        print(f"    RSA_ret NaN: {combined['RSA_r_ret'].isna().mean()*100:.1f}%")
        print(f"    RSA_enc NaN: {combined['RSA_r_enc'].isna().mean()*100:.1f}%")


# ============================================================================
# MAIN
# ============================================================================

if __name__ == '__main__':
    w2v_model = load_word2vec(WORD2VEC_PATH)

    for exp in EXPERIMENTS:
        run_experiment(exp, w2v_model)

    print(f"\n{'='*70}")
    print("✓ ALL EXPERIMENTS COMPLETE")
    print(f"{'='*70}")

  Loading Word2Vec from /home1/noaherz/word2vec/GoogleNews-vectors-negative300.bin.gz …
  ✓ Word2Vec loaded — vocab: 3,000,000

EXPERIMENT: DBOY1
  Loading subject_results_DBOY1/ALL_SUBJECTS_DBOY1_irasa_channel_wide.csv …
  Loaded 19,151 rows | 40 subjects | 8 sessions
  After HC filter: 12,346 rows
    Building semsim cache: 233 words → 27028 pairs …

  ────────────────────────────────────────────────────────────
  Region: LHP
  Rows: 7,218
    Processing trial group 0/179 …
    ✓ LHP/theta: 1,814 pairs | RSA_ret NaN=0.0% | RSA_enc NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_theta_rsa_lag.csv
    ✓ LHP/alpha: 1,814 pairs | RSA_ret NaN=16.7% | RSA_enc NaN=16.7% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_alpha_rsa_lag.csv
    ✓ LHP/beta: 1,814 pairs | RSA_ret NaN=0.0% | RSA_enc NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_beta_rsa_lag.csv
    ✓ LHP/gamma: 1,814 pairs | RSA_ret NaN=0.0% | RSA_enc NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_gamma_rsa_lag

In [3]:
#!/usr/bin/env python3
"""
LMM Analysis: T_lag / SP_lag → RSA_r_ret / RSA_r_enc
Hippocampus only (LHP / RHP), stratified by FREQUENCY BAND
----------------------------------------------------------------------
Adapted from analyze_iim.py for the rsa_lag_hc_bands.py output.

CHANGES FROM analyze_iim.py  (all marked with # ← CHANGED):
  1. OUTCOME       : split into RSA_r_ret and RSA_r_enc; loop runs both.
  2. REGIONS       : LHP, RHP only  (LLTC / RLTC removed)
  3. INPUT_DIR     : ./rsa_lag_hc_bands
  4. INPUT FILE    : ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag.csv
  5. OUTPUT_DIR    : ./rsa_lag_hc_bands_lmm
  6. PHASE         : removed — phase is encoded in column name (ret vs enc)
  7. Output tags   : include _ret / _enc suffix on all filenames/titles

Everything else — LMM formula, FDR, plotting functions — is UNCHANGED.
"""

import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, ttest_1samp
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENT  = 'DBOY1'
INPUT_DIR = Path('./rsa_lag_hc_bands_frac')
           
OUTPUT_DIR  = Path('./rsa_lag_hc_bands_lmm_frac')        # ← CHANGED
PLOT_DIR    = OUTPUT_DIR / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ← CHANGED: two outcomes instead of OUTCOME + PHASE
OUTCOMES    = ['RSA_r_ret', 'RSA_r_enc']
OUTCOME_LABELS = {
    'RSA_r_ret': 'Retrieval RSA',
    'RSA_r_enc': 'Encoding RSA',
}

PREDICTORS  = ['SP_lag', 'T_lag']
BANDS       = ['theta', 'alpha', 'beta', 'gamma']
REGIONS     = ['LHP', 'RHP']                        # ← CHANGED: HC only, no LLTC/RLTC

CROSS_COVARIATE = {
    'T_lag':  'SP_lag',
    'SP_lag': 'T_lag',
}
PRED_LABELS = {
    'T_lag':  'Temporal Lag (T_lag)',
    'SP_lag': 'Spatial Lag (SP_lag)',
}
MODEL_LABELS = {
    'Model1': 'Bare',
    'Model2': 'Controlled',
}

# ---- Palette ----------------------------------------------------------------
BG_COLOR    = 'white'
AX_COLOR    = '#F7F7F7'
GRID_COLOR  = '#DDDDDD'
TEXT_COLOR  = '#222222'
SPINE_COLOR = '#AAAAAA'

REGION_COLORS = {
    'LHP':  '#1A3A6B',
    'RHP':  '#8B1A1A',
}
BAND_COLORS = {
    'theta': '#4575B4',
    'alpha': '#74ADD1',
    'beta':  '#F46D43',
    'gamma': '#D73027',
}

# ============================================================================
# LMM FITTING  — outcome now passed as argument instead of global OUTCOME
# ============================================================================

def fit_lmm(df: pd.DataFrame,
            pred_cols: List[str],
            label: str,
            outcome: str,                           # ← CHANGED: was global OUTCOME
            formula_rhs: Optional[str] = None) -> Tuple[Optional[object], int]:
    df = df.copy()
    df['subj_sess'] = (df['subject'].astype(str)
                       + '_' + df['session'].astype(str))

    real_cols = [c for c in pred_cols if c in df.columns]
    keep      = [outcome] + real_cols + ['subject', 'subj_sess']  # ← CHANGED
    df        = df[keep].dropna()

    if len(df) < 20:
        print(f"    [{label}] Too few rows ({len(df)}) — skipping")
        return None, 0

    rhs     = formula_rhs if formula_rhs else ' + '.join(pred_cols)
    formula = f"{outcome} ~ {rhs}"                 # ← CHANGED
    print(f"    [{label}] {formula}  |  N={len(df):,}")

    model = MixedLM.from_formula(
        formula,
        data       = df,
        groups     = df['subject'],
        vc_formula = {'subj_sess': '0 + C(subj_sess)'},
    )

    result = None
    for method in ['lbfgs', 'nm', 'powell']:
        try:
            result = model.fit(reml=True, method=method)
            if np.isfinite(result.llf):
                print(f"    [{label}] optimizer={method}  "
                      f"converged={getattr(result, 'converged', None)}  "
                      f"llf={result.llf:.4f}  AIC={result.aic:.4f}")
                break
            else:
                print(f"    [{label}] llf=NaN with {method}, trying next …")
        except Exception as e:
            print(f"    [{label}] {method} failed: {e}")
            result = None

    if result is None or not np.isfinite(result.llf):
        print(f"    [{label}] WARNING: fit unsuccessful.")
    return result, len(df)


# ============================================================================
# RESULT EXTRACTION
# ============================================================================

def extract_rows(result,
                 pred_display: Dict[str, str],
                 model_label: str,
                 predictor: str,
                 band: str,
                 region: str,
                 outcome: str) -> pd.DataFrame:     # ← CHANGED: added outcome arg
    if result is None:
        return pd.DataFrame()
    rows = []
    for col, name in pred_display.items():
        matched = col if col in result.params.index else None
        if matched is None:
            hits    = [k for k in result.params.index
                       if col.lower() in k.lower()]
            matched = hits[0] if hits else None
        if matched is None:
            print(f"    WARNING: '{col}' not found in params — skipping")
            continue
        rows.append({
            'experiment':            EXPERIMENT,
            'outcome':               outcome,       # ← CHANGED: was 'phase'
            'region':                region,
            'predictor_of_interest': predictor,
            'band':                  band,
            'model':                 model_label,
            'term':                  name,
            'col':                   col,
            'beta':                  result.params[matched],
            'se':                    result.bse[matched],
            'z':                     result.tvalues[matched],
            'p_raw':                 result.pvalues[matched],
            'llf':                   result.llf,
            'aic':                   result.aic,
            'nobs':                  int(result.nobs),
        })
    return pd.DataFrame(rows)


def apply_fdr_within_model(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    _, df['p_fdr'] = fdrcorrection(df['p_raw'].values)
    return df


# ============================================================================
# TEXT FORMATTING
# ============================================================================

def sig_stars(p: float) -> str:
    return ('***' if p < 0.001 else
            '**'  if p < 0.01  else
            '*'   if p < 0.05  else
            '†'   if p < 0.10  else '')


def format_block(title: str, rows_df: pd.DataFrame, outcome: str) -> str:  # ← CHANGED
    sep  = '=' * 92
    sep2 = '-' * 92
    hdr  = (f"{'Term':<40} {'β':>8} {'SE':>8} {'z':>8} "
            f"{'p_raw':>10} {'p_fdr':>10} {'AIC':>10} {'N':>8} {'sig':>5}")
    lines = [sep, title, sep2, hdr, sep2]
    for _, row in rows_df.iterrows():
        aic_s = (f"{row['aic']:>10.2f}"
                 if pd.notna(row.get('aic')) else '       NaN')
        p_fdr = row.get('p_fdr', np.nan)
        lines.append(
            f"{row['term']:<40} {row['beta']:>8.4f} {row['se']:>8.4f} "
            f"{row['z']:>8.3f} {row['p_raw']:>10.4f} "
            f"{p_fdr:>10.4f} {aic_s} {int(row['nobs']):>8,} "
            f"{sig_stars(p_fdr):>5}"
        )
    lines += [sep2,
              'FDR: BH correction within each model across terms.',
              '† p<.10  * p<.05  ** p<.01  *** p<.001',
              f'Outcome = {outcome}.  Experiment = {EXPERIMENT}.',  # ← CHANGED
              'All continuous predictors on raw scale.',
              sep]
    return '\n'.join(lines)


# ============================================================================
# PLOTTING
# ============================================================================

def _style_ax(ax):
    ax.set_facecolor(AX_COLOR)
    ax.tick_params(colors=TEXT_COLOR, labelsize=8.5)
    ax.xaxis.label.set_color(TEXT_COLOR)
    ax.yaxis.label.set_color(TEXT_COLOR)
    ax.title.set_color(TEXT_COLOR)
    ax.grid(True, color=GRID_COLOR, lw=0.6, zorder=0)
    for spine in ax.spines.values():
        spine.set_edgecolor(SPINE_COLOR)
        spine.set_linewidth(0.8)


def plot_forest(all_results: pd.DataFrame,
                region: str,
                predictor: str,
                band: str,
                outcome: str,                       # ← CHANGED
                save_path: Path):
    df = all_results[
        (all_results['region']                == region) &
        (all_results['predictor_of_interest'] == predictor) &
        (all_results['band']                  == band) &
        (all_results['outcome']               == outcome) &  # ← CHANGED
        (all_results['col']                   == predictor)
    ].copy()

    if df.empty:
        return

    df['ci_lo'] = df['beta'] - 1.96 * df['se']
    df['ci_hi'] = df['beta'] + 1.96 * df['se']

    models = [m for m in ['Model1', 'Model2'] if m in df['model'].values]
    color  = REGION_COLORS.get(region, '#444444')

    fig, ax = plt.subplots(figsize=(9, max(3, len(models) * 1.4 + 1)))
    fig.patch.set_facecolor(BG_COLOR)
    _style_ax(ax)

    y_pos, yticks, ylabels = 0, [], []
    for model in models:
        row = df[df['model'] == model]
        if row.empty:
            continue
        row  = row.iloc[0]
        xerr = [[row['beta'] - row['ci_lo']], [row['ci_hi'] - row['beta']]]
        ax.errorbar(row['beta'], y_pos, xerr=xerr,
                    fmt='o', color=color, ecolor=color,
                    elinewidth=1.5, capsize=4, capthick=1.5,
                    markersize=7, zorder=3)
        p_show = row.get('p_fdr', row['p_raw'])
        stars  = sig_stars(p_show)
        if stars:
            offset = abs(row['ci_hi'] - row['beta']) * 0.15 + 0.001
            ax.text(row['ci_hi'] + offset, y_pos, stars,
                    color=color, va='center', fontsize=9, fontweight='bold')
        yticks.append(y_pos)
        ylabels.append(MODEL_LABELS.get(model, model))
        y_pos -= 1

    ax.axvline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=9)
    ax.set_xlabel(f'β  ({PRED_LABELS[predictor]})', fontsize=10)
    ax.set_title(
        f"{region}  |  {outcome} ~ {PRED_LABELS[predictor]}  |  "  # ← CHANGED
        f"{band.capitalize()} band  [{EXPERIMENT}]",
        fontsize=11, fontweight='bold', pad=8)

    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"    ✓ Forest → {save_path.name}")


def _subject_slopes(hd: pd.DataFrame, predictor: str, outcome: str) -> pd.DataFrame:  # ← CHANGED
    rows = []
    for subj, sg in hd.groupby('subject'):
        sg = sg.dropna(subset=[predictor, outcome])  # ← CHANGED
        if len(sg) < 3:
            continue
        x = sg[predictor].values.astype(float)
        y = sg[outcome].values.astype(float)         # ← CHANGED
        if x.std() == 0:
            continue
        m, b = np.polyfit(x, y, 1)
        rows.append({'subject': subj, 'slope': m,
                     'intercept': b, 'n_pairs': len(sg)})
    return pd.DataFrame(rows)


def _subject_zscore_rsa(hd: pd.DataFrame, outcome: str) -> pd.DataFrame:  # ← CHANGED
    hd = hd.copy()
    def zscore(x):
        s = x.std()
        return (x - x.mean()) / s if s > 0 else x - x.mean()
    hd[f'{outcome}_z'] = hd.groupby('subject')[outcome].transform(zscore)  # ← CHANGED
    return hd


def plot_interaction(region: str,
                     predictor: str,
                     band: str,
                     outcome: str,                  # ← CHANGED
                     raw_df: pd.DataFrame,
                     save_path: Path):
    outcome_z = f'{outcome}_z'                      # ← CHANGED

    sub_df = raw_df[
        raw_df['band'] == band
    ].copy().dropna(subset=[predictor, outcome])     # ← CHANGED

    if sub_df.empty:
        return

    color       = REGION_COLORS.get(region, '#444444')
    is_discrete = (predictor == 'T_lag')

    fig, (ax_sp, ax_sl) = plt.subplots(2, 1, figsize=(10, 10))
    fig.patch.set_facecolor(BG_COLOR)
    _style_ax(ax_sp)
    _style_ax(ax_sl)

    n_subj  = sub_df['subject'].nunique()
    n_pairs = len(sub_df)

    x_all = sub_df[predictor].values.astype(float)
    y_all = sub_df[outcome].values.astype(float)     # ← CHANGED
    r_val, p_val = pearsonr(x_all, y_all)
    p_str = f'p={p_val:.3f}' if p_val >= 0.001 else 'p<0.001'

    hd = _subject_zscore_rsa(sub_df, outcome)        # ← CHANGED

    if is_discrete:
        MAX_VALS = 20
        top_vals = sorted(
            hd[predictor].value_counts().nlargest(MAX_VALS).index.tolist())
        x_grid = np.array(top_vals, dtype=float)
    else:
        N_BINS     = 12
        hd['_bin'] = pd.cut(hd[predictor], bins=N_BINS)
        bin_mids   = (hd.groupby('_bin', observed=True)[predictor]
                       .mean().dropna())
        x_grid     = bin_mids.values

    subj_lines = []
    for subj, sg in hd.groupby('subject'):
        sg = sg.dropna(subset=[predictor, outcome_z])
        if len(sg) < 3:
            continue
        if is_discrete:
            pts = (sg.groupby(predictor)[outcome_z]
                     .mean().reindex(top_vals))
        else:
            pts = (sg.groupby('_bin', observed=True)[outcome_z].mean())
            pts.index = pts.index.map(
                lambda b: b.mid if hasattr(b, 'mid') else np.nan)
            pts = pts.dropna()
        pts = pts.dropna()
        if len(pts) < 2:
            continue
        subj_lines.append(pts.values)
        ax_sp.plot(x_grid[:len(pts.values)], pts.values,
                   color=color, alpha=0.15, lw=0.9, zorder=2)

    if subj_lines:
        max_len  = max(len(s) for s in subj_lines)
        padded   = np.array([
            np.pad(s.astype(float), (0, max_len - len(s)),
                   constant_values=np.nan)
            for s in subj_lines
        ])
        grp_mean = np.nanmean(padded, axis=0)
        grp_sem  = (np.nanstd(padded, axis=0)
                    / np.sqrt((~np.isnan(padded)).sum(axis=0)))
        xg = x_grid[:max_len]

        ax_sp.fill_between(xg, grp_mean - grp_sem, grp_mean + grp_sem,
                           color=color, alpha=0.20, zorder=3)
        ax_sp.plot(xg, grp_mean, color=color, lw=2.5, zorder=4,
                   label=f'Group mean ± SEM  (n={n_subj} subj)')

        valid = np.isfinite(grp_mean) & np.isfinite(xg)
        if valid.sum() >= 2:
            m_z, b_z = np.polyfit(xg[valid], grp_mean[valid], 1)
            ax_sp.plot(xg[valid], m_z * xg[valid] + b_z,
                       color=TEXT_COLOR, lw=1.5, ls='--', alpha=0.6,
                       zorder=5, label=f'OLS  r={r_val:.3f} {p_str}')

    ax_sp.axhline(0, color=SPINE_COLOR, lw=1, ls=':', zorder=1)
    if is_discrete:
        ax_sp.set_xticks(x_grid)
        ax_sp.set_xticklabels(
            [str(int(v)) for v in x_grid], fontsize=7,
            rotation=45 if len(x_grid) > 12 else 0)
    ax_sp.set_xlabel(PRED_LABELS[predictor], fontsize=9)
    ax_sp.set_ylabel(f'{outcome}  (z-scored within subject)', fontsize=9)  # ← CHANGED
    ax_sp.set_title(
        f'{region}  |  {band.capitalize()} band  |  {EXPERIMENT}\n'
        f'Outcome: {OUTCOME_LABELS[outcome]}  |  '                         # ← CHANGED
        f'n={n_subj} subj, {n_pairs:,} pairs',
        fontsize=10, fontweight='bold')
    ax_sp.legend(facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)

    slopes_df = _subject_slopes(hd, predictor, outcome)  # ← CHANGED

    if slopes_df.empty:
        ax_sl.set_visible(False)
    else:
        slopes     = slopes_df['slope'].values
        n_sl       = len(slopes)
        mean_slope = slopes.mean()
        sem_slope  = slopes.std() / np.sqrt(n_sl)
        ci95       = 1.96 * sem_slope
        n_neg      = (slopes < 0).sum()
        n_pos      = (slopes > 0).sum()

        order      = np.argsort(slopes)
        y_pos_sl   = np.arange(n_sl)
        dot_colors = [REGION_COLORS.get(region, '#444444')
                      if s >= 0 else '#888888'
                      for s in slopes[order]]

        ax_sl.scatter(slopes[order], y_pos_sl,
                      c=dot_colors, s=40, zorder=3,
                      edgecolors='white', linewidths=0.4)
        for yi, si, dc in zip(y_pos_sl, slopes[order], dot_colors):
            ax_sl.plot([0, si], [yi, yi],
                       color=dc, alpha=0.35, lw=0.8, zorder=2)

        ax_sl.axvline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
        ax_sl.axvspan(mean_slope - ci95, mean_slope + ci95,
                      color=color, alpha=0.12, zorder=0)
        ax_sl.axvline(mean_slope, color=color, lw=2.5, zorder=4,
                      label=f'Mean β={mean_slope:.4f} ± {ci95:.4f} (95%CI)')

        t_stat, p_1samp = ttest_1samp(slopes, 0)
        p1_str = f'p={p_1samp:.3f}' if p_1samp >= 0.001 else 'p<0.001'
        stars  = sig_stars(p_1samp)
        ax_sl.text(mean_slope, n_sl + 0.3,
                   f'{stars}  {p1_str}' if stars else p1_str,
                   ha='center', va='bottom',
                   color=color, fontsize=9, fontweight='bold')

        ax_sl.set_yticks([])
        ax_sl.set_xlabel(
            f'Subject-level OLS slope  ({outcome} ~ {predictor})', fontsize=9)  # ← CHANGED
        ax_sl.set_title(
            f'{region}  |  {band.capitalize()} band  |  {OUTCOME_LABELS[outcome]}\n'  # ← CHANGED
            f'Each dot = 1 subject  '
            f'({n_neg}/{n_sl} negative, {n_pos}/{n_sl} positive)',
            fontsize=10, fontweight='bold')
        ax_sl.legend(facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)
        pct_neg = n_neg / n_sl * 100
        ax_sl.text(0.02, 0.04,
                   f'{pct_neg:.0f}% of subjects show negative slope',
                   transform=ax_sl.transAxes,
                   fontsize=8, color=TEXT_COLOR, style='italic')

    fig.suptitle(
        f"{outcome} ~ {PRED_LABELS[predictor]}  |  {region}  |  "  # ← CHANGED
        f"{band.capitalize()} band\n"
        f"Top: z-scored spaghetti  |  Bottom: per-subject slopes",
        fontsize=12, fontweight='bold', y=1.01, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"    ✓ Interaction plot → {save_path.name}")


def plot_summary_heatmap(all_results: pd.DataFrame,
                         region: str,
                         outcome: str,              # ← CHANGED
                         save_path: Path):
    pivot_rows = []
    for pred in PREDICTORS:
        for band in BANDS:
            sub = all_results[
                (all_results['region']                == region) &
                (all_results['predictor_of_interest'] == pred) &
                (all_results['band']                  == band) &
                (all_results['outcome']               == outcome) &  # ← CHANGED
                (all_results['model']                 == 'Model1') &
                (all_results['col']                   == pred)
            ]
            beta  = sub.iloc[0]['beta']  if not sub.empty else np.nan
            stars = sig_stars(sub.iloc[0].get('p_fdr', sub.iloc[0]['p_raw'])) if not sub.empty else ''
            pivot_rows.append({'predictor': PRED_LABELS[pred],
                                'band': band, 'beta': beta, 'stars': stars})

    piv       = pd.DataFrame(pivot_rows)
    beta_mat  = piv.pivot(index='predictor', columns='band', values='beta')[BANDS]
    stars_mat = piv.pivot(index='predictor', columns='band', values='stars')[BANDS]

    fig, ax = plt.subplots(figsize=(9, 3.0))
    fig.patch.set_facecolor(BG_COLOR)
    vmax = np.nanmax(np.abs(beta_mat.values)) or 1.0
    im   = ax.imshow(beta_mat.values.astype(float),
                     aspect='auto', cmap='RdBu_r',
                     vmin=-vmax, vmax=vmax)

    for i in range(beta_mat.shape[0]):
        for j in range(beta_mat.shape[1]):
            val  = beta_mat.values[i, j]
            star = stars_mat.values[i, j]
            if not np.isnan(val):
                cell_norm = (val + vmax) / (2 * vmax)
                txt_color = 'white' if (cell_norm < 0.35 or cell_norm > 0.65) else TEXT_COLOR
                ax.text(j, i, f"{val:.3f}\n{star}",
                        ha='center', va='center',
                        color=txt_color, fontsize=9, fontweight='bold')

    ax.set_xticks(range(len(BANDS)))
    ax.set_xticklabels([b.capitalize() for b in BANDS], fontsize=10)
    ax.set_yticks(range(len(beta_mat.index)))
    ax.set_yticklabels(beta_mat.index.tolist(), fontsize=9)
    ax.tick_params(colors=TEXT_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(SPINE_COLOR)

    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label(f'β  (predictor → {outcome})', fontsize=9, color=TEXT_COLOR)  # ← CHANGED
    cbar.ax.yaxis.set_tick_params(color=TEXT_COLOR)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COLOR)

    ax.set_title(
        f'{region}  |  {OUTCOME_LABELS[outcome]} ~ predictor  |  '  # ← CHANGED
        f'β (Model 1)  [{EXPERIMENT}]',
        fontsize=10, fontweight='bold', pad=8, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"  ✓ Heatmap → {save_path.name}")


def plot_beta_bars(all_results: pd.DataFrame,
                   region: str,
                   outcome: str,                    # ← CHANGED
                   save_path: Path):
    n_cols = len(PREDICTORS)
    fig, axes = plt.subplots(2, n_cols, figsize=(7 * n_cols, 8))
    fig.patch.set_facecolor(BG_COLOR)
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    x     = np.arange(len(BANDS))
    width = 0.50

    for col_idx, pred in enumerate(PREDICTORS):
        for row_idx, model_key in enumerate(['Model1', 'Model2']):
            ax = axes[row_idx, col_idx]
            _style_ax(ax)

            betas, errors, pvals = [], [], []
            for band in BANDS:
                sub = all_results[
                    (all_results['region']                == region) &
                    (all_results['predictor_of_interest'] == pred) &
                    (all_results['outcome']               == outcome) &  # ← CHANGED
                    (all_results['model']                 == model_key) &
                    (all_results['band']                  == band) &
                    (all_results['col']                   == pred)
                ]
                if sub.empty:
                    betas.append(np.nan); errors.append(0); pvals.append(1.0)
                else:
                    r = sub.iloc[0]
                    betas.append(r['beta'])
                    errors.append(r['se'])
                    pvals.append(r.get('p_fdr', r['p_raw']))

            plot_betas  = [b if np.isfinite(b) else 0 for b in betas]
            plot_errors = [e if np.isfinite(betas[i]) else 0
                           for i, e in enumerate(errors)]

            bars = ax.bar(x, plot_betas, width,
                          color=[BAND_COLORS[b] for b in BANDS],
                          yerr=plot_errors,
                          error_kw=dict(ecolor=TEXT_COLOR, capsize=3, elinewidth=1),
                          alpha=0.82)

            for bar, beta, p in zip(bars, betas, pvals):
                if not np.isfinite(beta):
                    continue
                stars = sig_stars(p)
                if stars:
                    h    = bar.get_height()
                    sign = 1 if h >= 0 else -1
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            h + sign * max(abs(h) * 0.05, 0.005),
                            stars, ha='center', va='bottom',
                            color=TEXT_COLOR, fontsize=9, fontweight='bold')

            ax.axhline(0, color=SPINE_COLOR, lw=1.0, ls='--')
            ax.set_xticks(x)
            ax.set_xticklabels([b.capitalize() for b in BANDS], fontsize=9)
            ax.set_xlabel('Frequency Band', fontsize=9)
            ax.set_ylabel('β', fontsize=9)
            ax.set_title(
                f"{region}  |  {outcome} ~ {PRED_LABELS[pred]}\n"  # ← CHANGED
                f"[{MODEL_LABELS[model_key]}]",
                fontsize=9, fontweight='bold')

    fig.suptitle(
        f'{region}  |  {OUTCOME_LABELS[outcome]} ~ T_lag / SP_lag  |  '  # ← CHANGED
        f'β by Frequency Band  [{EXPERIMENT}]',
        fontsize=12, fontweight='bold', y=1.01, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"  ✓ Beta bars → {save_path.name}")


# ============================================================================
# DATA LOADING  — phase filter removed; new input filename
# ============================================================================

def load_data() -> Optional[pd.DataFrame]:
    # ← CHANGED: new input filename
    fpath = INPUT_DIR / f"ALL_SUBJECTS_{EXPERIMENT}_hc_allbands_rsa_lag.csv"

    if not fpath.exists():
        print(f"  ✗ Master CSV not found: {fpath}")
        print("  Trying per-region/band fallback …")
        dfs = []
        for region in REGIONS:
            for band in BANDS:
                p = INPUT_DIR / f"ALL_SUBJECTS_{EXPERIMENT}_{region}_{band}_rsa_lag.csv"
                if p.exists():
                    dfs.append(pd.read_csv(p))
        if not dfs:
            print("  ✗ No fallback files found either.")
            return None
        df = pd.concat(dfs, ignore_index=True)
    else:
        print(f"  Loading {fpath.name} …")
        df = pd.read_csv(fpath)
        print(f"  Loaded {len(df):,} rows")

    # ← CHANGED: no phase filter — phase encoded in RSA_r_ret / RSA_r_enc columns

    if 'region' in df.columns:
        df = df[df['region'].isin(REGIONS)].copy()
        print(f"  Region filter {REGIONS}: {len(df):,} rows")

    if df.empty:
        print("  ✗ No data after filtering.")
        return None

    df['subject'] = EXPERIMENT + '_' + df['subject'].astype(str)

    print(f"\n  Total rows   : {len(df):,}")
    print(f"  Subjects     : {df['subject'].nunique()}")
    print(f"  Regions      : {sorted(df['region'].unique().tolist())}")
    print(f"  Bands        : {sorted(df['band'].unique().tolist())}")
    return df


# ============================================================================
# ANALYSIS LOOP  — now iterates over OUTCOMES as well
# ============================================================================

def run_analysis_for_region_band(df: pd.DataFrame,
                                  region: str,
                                  predictor: str,
                                  band: str,
                                  outcome: str,     # ← CHANGED
                                  has_semsim: bool
                                  ) -> Tuple[pd.DataFrame, str]:
    sub = df[
        (df['region'] == region) &
        (df['band']   == band)
    ].copy()

    if sub.empty:
        return pd.DataFrame(), ""

    print(f"\n  ── {region}  |  {outcome} ~ {predictor}  |  {band.capitalize()} ──")  # ← CHANGED
    print(f"     Rows: {len(sub):,}  |  Subjects: {sub['subject'].nunique()}")

    all_rows    = []
    text_blocks = [
        f"EXPERIMENT: {EXPERIMENT}  OUTCOME: {outcome}  "         # ← CHANGED
        f"REGION: {region}  PREDICTOR: {predictor}  BAND: {band}",
        f"N rows: {len(sub):,}  |  subjects: {sub['subject'].nunique()}\n",
    ]

    cross_cov   = CROSS_COVARIATE[predictor]
    cross_label = PRED_LABELS[cross_cov]

    def run_and_collect(real_cols, model_key, title, pred_display,
                        formula_rhs=None):
        res, _ = fit_lmm(sub, real_cols, model_key, outcome,     # ← CHANGED
                         formula_rhs=formula_rhs)
        rows   = extract_rows(res, pred_display, model_key,
                              predictor, band, region, outcome)   # ← CHANGED
        rows   = apply_fdr_within_model(rows)
        if not rows.empty:
            all_rows.append(rows)
            block = format_block(title, rows, outcome)            # ← CHANGED
            text_blocks.append(block)
            print('\n' + block)
        return res

    # Model 1: bare
    run_and_collect(
        [predictor],
        'Model1',
        f'Model 1 — {outcome} ~ {predictor}  [{region} / {band}]',  # ← CHANGED
        {predictor: PRED_LABELS[predictor]},
    )

    # Model 2: controlled
    ctrl_cols = [predictor, cross_cov, 'output_lag']
    ctrl_disp = {
        predictor:    PRED_LABELS[predictor],
        cross_cov:    f'{cross_label}  [covariate]',
        'output_lag': 'output_lag',
    }
    if has_semsim:
        ctrl_cols = [predictor, cross_cov, 'output_lag', 'semantic_sim']
        ctrl_disp['semantic_sim'] = 'semantic_sim'

    run_and_collect(
        ctrl_cols,
        'Model2',
        f'Model 2 — {outcome} ~ {predictor} + {cross_cov} + controls  '  # ← CHANGED
        f'[{region} / {band}]',
        ctrl_disp,
    )

    result_df = (pd.concat(all_rows, ignore_index=True)
                 if all_rows else pd.DataFrame())
    return result_df, '\n\n'.join(text_blocks)


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*80}")
    print(f"RSA ~ T_lag / SP_lag  |  HC only  |  {EXPERIMENT}")
    print(f"Outcomes: {OUTCOMES}")                  # ← CHANGED
    print(f"{'='*80}")

    df = load_data()
    if df is None or df.empty:
        print("No data loaded. Exiting.")
        return

    has_semsim = ('semantic_sim' in df.columns
                  and df['semantic_sim'].notna().any())
    print(f"\n  Outcomes       : {OUTCOMES}")       # ← CHANGED
    print(f"  Predictors     : {PREDICTORS}")
    print(f"  Bands          : {BANDS}")
    print(f"  Regions        : {REGIONS}")
    print(f"  semantic_sim   : {has_semsim}")

    ALL_RESULTS = []

    # ← CHANGED: outer loop over outcomes
    for outcome in OUTCOMES:
        print(f"\n{'━'*80}")
        print(f"OUTCOME: {outcome}  ({OUTCOME_LABELS[outcome]})")
        print(f"{'━'*80}")

        for region in REGIONS:
            region_results = []
            print(f"\n{'─'*80}")
            print(f"REGION: {region}")
            print(f"{'─'*80}")

            for predictor in PREDICTORS:
                for band in BANDS:
                    res_df, text = run_analysis_for_region_band(
                        df, region, predictor, band, outcome, has_semsim)

                    if not res_df.empty:
                        region_results.append(res_df)
                        ALL_RESULTS.append(res_df)

                    # ← CHANGED: tag includes outcome suffix
                    tag      = f"{region}_{predictor}_{band}_{outcome}"
                    csv_path = OUTPUT_DIR / f"LMM_{tag}_results.csv"
                    txt_path = OUTPUT_DIR / f"LMM_{tag}_results.txt"
                    if not res_df.empty:
                        res_df.to_csv(csv_path, index=False)
                        print(f"    ✓ {csv_path.name}")
                    if text:
                        with open(txt_path, 'w') as f:
                            f.write(text)
                        print(f"    ✓ {txt_path.name}")

            # Per-region plots — one set per outcome              # ← CHANGED
            if region_results:
                reg_df = pd.concat(region_results, ignore_index=True)

                plot_summary_heatmap(
                    reg_df, region, outcome,
                    PLOT_DIR / f"summary_heatmap_{region}_{outcome}.png")   # ← CHANGED
                plot_beta_bars(
                    reg_df, region, outcome,
                    PLOT_DIR / f"beta_bars_{region}_{outcome}.png")         # ← CHANGED

                for predictor in PREDICTORS:
                    for band in BANDS:
                        region_sub = df[
                            (df['region'] == region) &
                            (df['band']   == band)
                        ].copy()

                        plot_forest(
                            reg_df, region, predictor, band, outcome,
                            PLOT_DIR / f"forest_{region}_{predictor}_{band}_{outcome}.png")
                        plot_interaction(
                            region, predictor, band, outcome, region_sub,
                            PLOT_DIR / f"interaction_{region}_{predictor}_{band}_{outcome}.png")

    if ALL_RESULTS:
        master_df   = pd.concat(ALL_RESULTS, ignore_index=True)
        master_path = OUTPUT_DIR / 'LMM_ALL_results.csv'
        master_df.to_csv(master_path, index=False)
        print(f"\n  ✓ Master results → {master_path.name}  "
              f"({len(master_df):,} rows)")

    print(f"\n{'='*80}")
    print(f"✓ ANALYSIS COMPLETE  [{EXPERIMENT}]")
    print(f"  Results : {OUTPUT_DIR}")
    print(f"  Plots   : {PLOT_DIR}")
    print(f"{'='*80}")


if __name__ == '__main__':
    main()


RSA ~ T_lag / SP_lag  |  HC only  |  DBOY1
Outcomes: ['RSA_r_ret', 'RSA_r_enc']
  Loading ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag.csv …
  Loaded 13,028 rows
  Region filter ['LHP', 'RHP']: 13,028 rows

  Total rows   : 13,028
  Subjects     : 34
  Regions      : ['LHP', 'RHP']
  Bands        : ['alpha', 'beta', 'gamma', 'theta']

  Outcomes       : ['RSA_r_ret', 'RSA_r_enc']
  Predictors     : ['SP_lag', 'T_lag']
  Bands          : ['theta', 'alpha', 'beta', 'gamma']
  Regions        : ['LHP', 'RHP']
  semantic_sim   : True

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTCOME: RSA_r_ret  (Retrieval RSA)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

────────────────────────────────────────────────────────────────────────────────
REGION: LHP
────────────────────────────────────────────────────────────────────────────────

  ── LHP  |  RSA_r_ret ~ SP_lag  |  Theta ──
     Rows: 1,814  |  Subjects: 27
    [Model1] RSA

In [3]:
#!/usr/bin/env python3
"""
RSA Lag Analysis — Hippocampus Only (LHP / RHP), Band × Pair (Long Format)
---------------------------------------------------------------------------
*** MIXED POWER variant (frac + osc → mixed, then band-split RSA) ***

Pipeline per recall event:
  Step 1 — Reconstruct mixed power per channel per frequency:
               mixed[ch, freq] = ret_frac_f{freq} + ret_osc_f{freq}

  Step 2 — Slice to band frequency indices → (N_band_freqs × N_channels) matrix
            (freq is the OUTER / row dimension, channel is the INNER / col dimension)

  Step 3 — Flatten row-major → 1-D vector of length (N_band_freqs × N_channels)

  Step 4 — RSA_r = Pearson r between vectors of recall i and recall j
            (NaN elements masked pairwise; require ≥ 3 finite pairs)

Regions : LHP, RHP
Bands   : theta  (freq idx 0–3,   ~3–6 Hz)
          alpha  (freq idx 4–6,   ~7–11 Hz)
          beta   (freq idx 7–11,  ~14–34 Hz)
          gamma  (freq idx 12–17, ~42–128 Hz)

18 log-spaced frequencies (3–128 Hz):
  idx:  0     1     2     3     4     5      6      7      8
  Hz:  3.00  3.74  4.67  5.82  7.26  9.05  11.28  14.07  17.55
  idx:  9     10    11    12    13    14    15     16     17
  Hz: 21.88 27.29 34.03 42.44 52.92 66.00 82.31 102.64 128.00

Output — LONG format: one row per pair × band.

Columns:
  subject, session, experiment, trial,
  region, hemisphere,
  band, band_freq_indices,
  output_pos_i, output_pos_j, output_lag,
  word_i, word_j,
  serial_pos_i, serial_pos_j, T_lag, SP_lag,
  RSA_r, n_channels, semantic_sim

Output files:
  ./rsa_lag_hc_bands_mixed/ALL_SUBJECTS_{exp}_{region}_{band}_rsa_lag_mixed.csv
  ./rsa_lag_hc_bands_mixed/ALL_SUBJECTS_{exp}_{region}_allbands_rsa_lag_mixed.csv
  ./rsa_lag_hc_bands_mixed/ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag_mixed.csv
"""

import warnings
import traceback
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS = ['DBOY1', 'EFRCourierReadOnly', 'EFRCourierOpenLoop']

INPUT_DIRS = {
    'DBOY1':               Path('./subject_results_DBOY1'),
    'EFRCourierReadOnly':  Path('./subject_results_EFRCourierReadOnly'),
    'EFRCourierOpenLoop':  Path('./subject_results_EFRCourierOpenLoop'),
}

OUTPUT_DIR = Path('./rsa_lag_hc_bands_mixed')
OUTPUT_DIR.mkdir(exist_ok=True)

# Hippocampus only
REGIONS    = ['LHP', 'RHP']
HEMISPHERE = {'LHP': 'left', 'RHP': 'right'}

# 18 log-spaced frequencies 3–128 Hz
N_FREQS = 18

BANDS = {
    'theta': list(range(0, 4)),    # 3.00 – 5.82 Hz
    'alpha': list(range(4, 7)),    # 7.26 – 11.28 Hz
    'beta':  list(range(7, 12)),   # 14.07 – 34.03 Hz
    'gamma': list(range(12, 18)),  # 42.44 – 128.00 Hz
}
BAND_ORDER = ['theta', 'alpha', 'beta', 'gamma']

# Source columns (fractal and oscillatory SSL power)
RET_FRAC_COLS = [f'ret_frac_f{i:02d}' for i in range(N_FREQS)]
RET_OSC_COLS  = [f'ret_osc_f{i:02d}'  for i in range(N_FREQS)]

# ---- Word2Vec ---------------------------------------------------------------
WORD2VEC_PATH = Path('/home1/noaherz/word2vec/GoogleNews-vectors-negative300.bin.gz')

OUTPUT_COLS = [
    'subject', 'session', 'experiment', 'trial',
    'region', 'hemisphere',
    'band', 'band_freq_indices',
    'output_pos_i', 'output_pos_j', 'output_lag',
    'word_i', 'word_j',
    'serial_pos_i', 'serial_pos_j', 'T_lag', 'SP_lag',
    'RSA_r', 'n_channels', 'semantic_sim',
]

# ============================================================================
# WORD2VEC LOADER + SIMILARITY
# ============================================================================

def load_word2vec(path: Path):
    if path is None or not path.exists():
        print(f"  ⚠  Word2Vec model not found at {path}. semantic_sim will be NaN.")
        return None
    try:
        import gensim.models as gensim_models
        print(f"  Loading Word2Vec from {path} …")
        model = gensim_models.KeyedVectors.load_word2vec_format(str(path), binary=True)
        try:
            vs = len(model)
        except TypeError:
            vs = len(model.vocab)
        print(f"  ✓ Word2Vec loaded — vocab: {vs:,}")
        return model
    except Exception as e:
        print(f"  ✗ Word2Vec load failed: {e}. semantic_sim will be NaN.")
        return None


def case_insensitive_similarity(word1: str, word2: str, model) -> Optional[float]:
    cases = [
        (word1.lower(), word2.lower()),
        (word1.lower(), word2.upper()),
        (word1.upper(), word2.lower()),
        (word1.upper(), word2.upper()),
    ]
    sims = []
    for w1, w2 in cases:
        try:
            sims.append(model.similarity(w1, w2))
        except KeyError:
            continue
    return float(max(sims)) if sims else None


def build_similarity_cache(words: set, model) -> dict:
    if model is None:
        return {}
    unique_words = sorted(w for w in words if isinstance(w, str))
    n = len(unique_words)
    print(f"    Building semsim cache: {n} words → {n*(n-1)//2} pairs …")
    cache = {}
    for i, w1 in enumerate(unique_words):
        for w2 in unique_words[i:]:
            key = frozenset({w1, w2})
            if key not in cache:
                sim = case_insensitive_similarity(w1, w2, model)
                cache[key] = sim if sim is not None else np.nan
    return cache


# ============================================================================
# VECTOR + STATISTICS HELPERS
# ============================================================================

def build_mixed_band_vector(recall_rows: pd.DataFrame,
                            channel_index,
                            band_freq_indices: list) -> np.ndarray:
    """
    Build the representational vector for one recall event and one band.

    Steps:
      1. Align rows to canonical channel_index → (N_ch,) indexed DataFrame
      2. Compute mixed = frac + osc for all 18 frequencies
         → mixed_mat shape: (N_ch, N_freqs)
      3. Slice to band_freq_indices
         → band_mat shape: (N_ch, N_band_freqs)
      4. Transpose to (N_band_freqs, N_ch)   ← freq is outer dimension
      5. Flatten row-major → 1-D vector of length N_band_freqs * N_ch

    Missing channels (not in recall_rows) retain NaN — handled in safe_pearsonr.
    """
    # Step 1: align to canonical channel order
    ch_df = (
        recall_rows
        .drop_duplicates(subset='channel_idx')
        .set_index('channel_idx')
        .reindex(channel_index)
    )

    # Step 2: mixed power = frac + osc, shape (N_ch, N_freqs)
    frac_mat = ch_df[RET_FRAC_COLS].values   # (N_ch, 18)
    osc_mat  = ch_df[RET_OSC_COLS].values    # (N_ch, 18)
    mixed_mat = frac_mat + osc_mat           # (N_ch, 18)  — element-wise sum

    # Step 3: slice to band frequencies → (N_ch, N_band_freqs)
    band_mat = mixed_mat[:, band_freq_indices]

    # Step 4: transpose → (N_band_freqs, N_ch)
    band_mat_T = band_mat.T

    # Step 5: flatten row-major → (N_band_freqs * N_ch,)
    return band_mat_T.flatten()


def safe_pearsonr(v1: np.ndarray, v2: np.ndarray) -> float:
    if len(v1) != len(v2):
        return np.nan
    mask = np.isfinite(v1) & np.isfinite(v2)
    if mask.sum() < 3:
        return np.nan
    r, _ = pearsonr(v1[mask], v2[mask])
    return float(r)


def safe_euclidean(x1, z1, x2, z2) -> float:
    if any(not np.isfinite(v) for v in (x1, z1, x2, z2)):
        return np.nan
    return float(euclidean([x1, z1], [x2, z2]))


def extract_scalar(series: pd.Series, field: str, context: str):
    unique_vals = series.dropna().unique()
    if len(unique_vals) > 1:
        warnings.warn(
            f"[{context}] '{field}' has {len(unique_vals)} distinct values "
            f"({unique_vals[:3]}…). Taking first.")
    return series.iloc[0]


# ============================================================================
# TRIAL-LEVEL PROCESSING
# ============================================================================

def process_trial_region(trial_df: pd.DataFrame,
                         region: str,
                         sim_cache: dict) -> List[Dict]:
    """
    For one (subject, session, trial, region):
      - Enumerate all valid recall pairs (i, j), i < j
      - For each pair × each band: compute RSA_r on the mixed-power band vector
      - Returns list of dicts (long format, one row per pair × band)
    """
    rows = []

    output_positions = sorted(
        trial_df['recall_output_position'].unique(),
        key=lambda x: int(x),
    )
    if len(output_positions) < 2:
        return rows

    channel_index = sorted(trial_df['channel_idx'].unique(), key=int)
    sample_row    = trial_df.iloc[0]

    # ------------------------------------------------------------------
    # Pre-compute per-output-position metadata + per-band mixed vectors
    # ------------------------------------------------------------------
    pos_data: Dict[int, Dict] = {}

    for op in output_positions:
        op_rows = trial_df[trial_df['recall_output_position'] == op]
        ctx = (f"subj={sample_row['subject']} sess={sample_row['session']} "
               f"trial={sample_row['trial']} region={region} op={op}")

        word       = extract_scalar(op_rows['recalled_word'],   'recalled_word',   ctx)
        serial_pos = extract_scalar(op_rows['serial_position'], 'serial_position', ctx)
        store_x    = extract_scalar(op_rows['store_x'],         'store_x',         ctx)
        store_z    = extract_scalar(op_rows['store_z'],         'store_z',         ctx)
        n_ch       = op_rows['channel_idx'].nunique()

        # Build one mixed-power vector per band
        band_vectors = {
            band_name: build_mixed_band_vector(op_rows, channel_index, freq_idx)
            for band_name, freq_idx in BANDS.items()
        }

        pos_data[op] = {
            'band_vectors': band_vectors,
            'word':         word,
            'serial_pos':   serial_pos,
            'store_x':      store_x,
            'store_z':      store_z,
            'n_channels':   n_ch,
        }

    subject    = sample_row['subject']
    session    = sample_row['session']
    experiment = sample_row['experiment']
    trial      = sample_row['trial']

    # ------------------------------------------------------------------
    # Enumerate pairs × bands
    # ------------------------------------------------------------------
    for idx_i, op_i in enumerate(output_positions):
        for op_j in output_positions[idx_i + 1:]:
            d_i = pos_data[op_i]
            d_j = pos_data[op_j]

            output_lag = int(op_j) - int(op_i)
            T_lag      = abs(int(d_i['serial_pos']) - int(d_j['serial_pos']))
            SP_lag     = safe_euclidean(d_i['store_x'], d_i['store_z'],
                                        d_j['store_x'], d_j['store_z'])
            n_channels = min(d_i['n_channels'], d_j['n_channels'])

            # Semantic similarity — same for all bands, compute once per pair
            w_i = str(d_i['word']).lower() if pd.notna(d_i['word']) else None
            w_j = str(d_j['word']).lower() if pd.notna(d_j['word']) else None
            sem_sim = (
                sim_cache.get(frozenset({w_i, w_j}), np.nan)
                if (w_i and w_j and sim_cache)
                else np.nan
            )

            # One row per band
            for band_name in BAND_ORDER:
                freq_idx = BANDS[band_name]
                v_i      = d_i['band_vectors'][band_name]
                v_j      = d_j['band_vectors'][band_name]
                rsa_r    = safe_pearsonr(v_i, v_j)

                rows.append({
                    'subject':           subject,
                    'session':           session,
                    'experiment':        experiment,
                    'trial':             trial,
                    'region':            region,
                    'hemisphere':        HEMISPHERE[region],
                    'band':              band_name,
                    'band_freq_indices': str(freq_idx),
                    'output_pos_i':      op_i,
                    'output_pos_j':      op_j,
                    'output_lag':        output_lag,
                    'word_i':            d_i['word'],
                    'word_j':            d_j['word'],
                    'serial_pos_i':      d_i['serial_pos'],
                    'serial_pos_j':      d_j['serial_pos'],
                    'T_lag':             T_lag,
                    'SP_lag':            SP_lag,
                    'RSA_r':             rsa_r,
                    'n_channels':        n_channels,
                    'semantic_sim':      sem_sim,
                })

    return rows


# ============================================================================
# PER-EXPERIMENT RUNNER
# ============================================================================

def run_experiment(exp: str, w2v_model):
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {exp}")
    print(f"{'='*70}")

    input_dir  = INPUT_DIRS.get(exp)
    if input_dir is None:
        print(f"  ✗ No INPUT_DIR configured for '{exp}'.")
        return

    input_path = input_dir / f"ALL_SUBJECTS_{exp}_irasa_channel_wide.csv"
    if not input_path.exists():
        print(f"  ✗ Master CSV not found: {input_path}")
        return

    print(f"  Loading {input_path} …")
    df = pd.read_csv(input_path)
    print(f"  Loaded {len(df):,} rows | "
          f"{df['subject'].nunique()} subjects | "
          f"{df['session'].nunique()} sessions")

    # Verify required columns exist
    missing_frac = [c for c in RET_FRAC_COLS if c not in df.columns]
    missing_osc  = [c for c in RET_OSC_COLS  if c not in df.columns]
    if missing_frac or missing_osc:
        print(f"  ✗ Missing columns — frac: {missing_frac[:3]}{'…' if len(missing_frac)>3 else ''} "
              f"| osc: {missing_osc[:3]}{'…' if len(missing_osc)>3 else ''}")
        return

    # Filter to HC only
    df = df[df['region'].isin(REGIONS)].copy()
    print(f"  After HC filter: {len(df):,} rows")
    if df.empty:
        print("  ✗ No hippocampal data found — check 'region' column values.")
        return

    df['channel_idx'] = df['channel_idx'].astype(int)

    # Build Word2Vec cache once per experiment
    all_words_lower = set(
        df['recalled_word'].dropna().astype(str).str.lower().unique()
    )
    sim_cache = build_similarity_cache(all_words_lower, w2v_model)

    all_region_dfs = []

    for region in REGIONS:
        print(f"\n  {'─'*60}")
        print(f"  Region: {region}")

        region_df = df[df['region'] == region].copy()
        if region_df.empty:
            print(f"  ✗ No rows for region {region} — skipping")
            continue

        print(f"  Rows: {len(region_df):,}")

        all_rows = []
        groups   = region_df.groupby(['subject', 'session', 'trial'])
        n_groups = len(groups)

        for g_idx, ((subj, sess, trial), trial_df) in enumerate(groups):
            if g_idx % 200 == 0:
                print(f"    Processing trial group {g_idx}/{n_groups} …")
            try:
                rows = process_trial_region(trial_df, region, sim_cache)
                all_rows.extend(rows)
            except Exception as e:
                print(f"    FAILED [{subj} sess={sess} trial={trial}]: {e}")
                traceback.print_exc()
                continue

        if not all_rows:
            print(f"  ✗ No pairs generated for region {region}")
            continue

        result_df = pd.DataFrame(all_rows, columns=OUTPUT_COLS)
        all_region_dfs.append(result_df)

        # Per-region × per-band outputs
        for band_name in BAND_ORDER:
            band_df  = result_df[result_df['band'] == band_name]
            out_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_{region}_{band_name}_rsa_lag_mixed.csv"
            band_df.to_csv(out_path, index=False)
            print(f"    ✓ {region}/{band_name}: {len(band_df):,} pairs | "
                  f"RSA NaN={band_df['RSA_r'].isna().mean()*100:.1f}% | "
                  f"SemSim NaN={band_df['semantic_sim'].isna().mean()*100:.1f}% | "
                  f"→ {out_path.name}")

        # All-bands file for this region
        reg_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_{region}_allbands_rsa_lag_mixed.csv"
        result_df.to_csv(reg_path, index=False)
        print(f"  ✓ {region} all-bands → {reg_path.name}")

    # Combined: both HC regions × all bands
    if all_region_dfs:
        combined  = pd.concat(all_region_dfs, ignore_index=True)
        comb_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag_mixed.csv"
        combined.to_csv(comb_path, index=False)
        print(f"\n  ✓ Combined HC all-bands → {comb_path.name}")
        print(f"    Total rows : {len(combined):,}")
        print(f"    Regions    : {combined['region'].unique().tolist()}")
        print(f"    Bands      : {combined['band'].unique().tolist()}")


# ============================================================================
# MAIN
# ============================================================================

if __name__ == '__main__':
    w2v_model = load_word2vec(WORD2VEC_PATH)

    for exp in EXPERIMENTS:
        run_experiment(exp, w2v_model)

    print(f"\n{'='*70}")
    print("✓ ALL EXPERIMENTS COMPLETE")
    print(f"{'='*70}")

  Loading Word2Vec from /home1/noaherz/word2vec/GoogleNews-vectors-negative300.bin.gz …
  ✓ Word2Vec loaded — vocab: 3,000,000

EXPERIMENT: DBOY1
  Loading subject_results_DBOY1/ALL_SUBJECTS_DBOY1_irasa_channel_wide.csv …
  Loaded 9,298 rows | 37 subjects | 6 sessions
  After HC filter: 6,173 rows
    Building semsim cache: 233 words → 27028 pairs …

  ────────────────────────────────────────────────────────────
  Region: LHP
  Rows: 3,609
    Processing trial group 0/179 …
    ✓ LHP/theta: 1,814 pairs | RSA NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_theta_rsa_lag_mixed.csv
    ✓ LHP/alpha: 1,814 pairs | RSA NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_alpha_rsa_lag_mixed.csv
    ✓ LHP/beta: 1,814 pairs | RSA NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_beta_rsa_lag_mixed.csv
    ✓ LHP/gamma: 1,814 pairs | RSA NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_gamma_rsa_lag_mixed.csv
  ✓ LHP all-bands → ALL_SUBJECTS_DBOY1_LHP_allbands_rsa_lag_mixed.c

In [4]:
#!/usr/bin/env python3
"""
Plot RSA Lag LMM Results
------------------------
Two figures:
  Figure 1 — T_lag predictor
  Figure 2 — SP_lag predictor

Each figure layout:
  Row 1 : Model 1 (bare)      — T_lag/SP_lag beta, LHP vs RHP side-by-side per band
  Row 2 : Model 2 (controlled)— predictor beta + covariate beta overlaid, LHP vs RHP

  Col 1 : Encoding  (RSA_r_enc)
  Col 2 : Retrieval (RSA_r_ret)

Color scheme (consistent across all plots):
  LHP          : dark blue   #1A3A6B
  RHP          : dark red    #8B1A1A
  Covariate    : brown       #6B3A1A
  Significance : * p<.05 (uncorrected), raw p values only

Error bars : SE from LMM
Background : white with dark text/spines
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

try:
    matplotlib.use('TkAgg')
except Exception:
    matplotlib.use('Agg')   # fallback for headless environments

# ============================================================================
# CONFIGURATION — match analyze_iim_hc_bands.py
# ============================================================================

EXPERIMENT  = 'DBOY1'
INPUT_DIR   = Path('./rsa_lag_hc_bands_lmm_frac')
OUTPUT_DIR  = Path('./rsa_lag_hc_bands_lmm_frac/plots')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BANDS       = ['theta', 'alpha', 'beta', 'gamma']
REGIONS     = ['LHP', 'RHP']
OUTCOMES    = ['RSA_r_enc', 'RSA_r_ret']          # enc first (col 1), ret second (col 2)
PREDICTORS  = ['T_lag', 'SP_lag']
CROSS_COVARIATE = {'T_lag': 'SP_lag', 'SP_lag': 'T_lag'}

OUTCOME_LABELS  = {'RSA_r_enc': 'Encoding', 'RSA_r_ret': 'Retrieval'}
PRED_LABELS     = {'T_lag': 'Temporal Lag', 'SP_lag': 'Spatial Lag'}
CROSS_LABELS    = {'SP_lag': 'SP lag (ctrl)', 'T_lag': 'T lag (ctrl)',
                   'output_lag': 'Output lag (ctrl)', 'semantic_sim': 'Sem sim (ctrl)'}

# ---- Consistent color palette -----------------------------------------------
COLOR_LHP       = '#1A3A6B'   # dark blue
COLOR_RHP       = '#8B1A1A'   # dark red
COLOR_CROSS     = '#6B3A1A'   # brown  (cross-covariate)
COLOR_OUTLG     = '#1A4010'   # very dark green (output_lag covariate)
COLOR_SEMSIM    = '#5A1A6B'   # dark purple (semantic_sim covariate)
COLOR_ERR       = '#222222'   # error bar colour
BG_COLOR        = 'white'
AX_COLOR        = '#F5F5F5'
GRID_COLOR      = '#DDDDDD'
TEXT_COLOR      = '#111111'
SPINE_COLOR     = '#888888'

COVARIATE_COLORS = {
    'SP_lag':       COLOR_CROSS,
    'T_lag':        COLOR_CROSS,
    'output_lag':   COLOR_OUTLG,
    'semantic_sim': COLOR_SEMSIM,
}

# ---- Font sizes -------------------------------------------------------------
TITLE_FS    = 15
SUPTITLE_FS = 17
LABEL_FS    = 13
TICK_FS     = 11
LEGEND_FS   = 12
ANNOT_FS    = 13   # significance stars

BAR_WIDTH   = 0.32
BAR_GAP     = 0.04   # gap between LHP and RHP within a band

# ============================================================================
# HELPERS
# ============================================================================

def _style_ax(ax):
    ax.set_facecolor(AX_COLOR)
    ax.tick_params(colors=TEXT_COLOR, labelsize=TICK_FS)
    ax.xaxis.label.set_color(TEXT_COLOR)
    ax.yaxis.label.set_color(TEXT_COLOR)
    ax.title.set_color(TEXT_COLOR)
    ax.grid(True, axis='y', color=GRID_COLOR, lw=0.7, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor(SPINE_COLOR)
        spine.set_linewidth(0.9)


def sig_star(p: float) -> str:
    """Uncorrected significance stars."""
    if np.isnan(p):
        return ''
    return ('***' if p < 0.001 else
            '**'  if p < 0.01  else
            '*'   if p < 0.05  else '')


def _annotate_bar(ax, x_center: float, bar_top: float, star: str, color: str):
    if not star:
        return
    offset = abs(bar_top) * 0.06 + 0.002
    y      = bar_top + offset if bar_top >= 0 else bar_top - offset
    va     = 'bottom' if bar_top >= 0 else 'top'
    ax.text(x_center, y, star,
            ha='center', va=va,
            color=color, fontsize=ANNOT_FS, fontweight='bold', zorder=6)


def load_master(experiment: str) -> pd.DataFrame:
    path = INPUT_DIR / 'LMM_ALL_results.csv'
    if not path.exists():
        raise FileNotFoundError(f"Master CSV not found: {path}")
    df = pd.read_csv(path)
    df = df[df['experiment'].str.startswith(experiment)].copy()
    print(f"Loaded {len(df):,} rows for experiment '{experiment}'")
    return df


def get_beta(df: pd.DataFrame,
             region: str, band: str, outcome: str,
             predictor_of_interest: str, model: str, col: str):
    """
    Extract (beta, se, p_raw) for a specific cell.
    Returns (nan, nan, nan) if not found.
    """
    sub = df[
        (df['region']                == region) &
        (df['band']                  == band) &
        (df['outcome']               == outcome) &
        (df['predictor_of_interest'] == predictor_of_interest) &
        (df['model']                 == model) &
        (df['col']                   == col)
    ]
    if sub.empty:
        return np.nan, np.nan, np.nan
    r = sub.iloc[0]
    return float(r['beta']), float(r['se']), float(r['p_raw'])


# ============================================================================
# PANEL DRAWING
# ============================================================================

def draw_model1_panel(ax, df: pd.DataFrame,
                      predictor: str, outcome: str):
    """
    Model 1 (bare): mean ± SE errorbar plot.
    LHP and RHP shown side-by-side at each band position.
    """
    x_centers = np.arange(len(BANDS), dtype=float)

    offsets = {
        'LHP': -0.15,
        'RHP':  0.15,
    }
    colors = {'LHP': COLOR_LHP, 'RHP': COLOR_RHP}

    # collect all beta±se extremes to set y-limits with star headroom
    y_extremes = []

    for region in REGIONS:
        betas, ses, ps = [], [], []
        for band in BANDS:
            b, s, p = get_beta(df, region, band, outcome,
                               predictor, 'Model1', predictor)
            betas.append(b); ses.append(s); ps.append(p)

        betas = np.array(betas, dtype=float)
        ses   = np.array(ses,   dtype=float)
        xpos  = x_centers + offsets[region]
        color = colors[region]

        ax.errorbar(xpos, betas,
                    yerr=np.where(np.isfinite(ses), ses, 0),
                    fmt='o', color=color,
                    ecolor=color, elinewidth=2.0,
                    capsize=5, capthick=2.0,
                    markersize=8, markeredgewidth=1.2,
                    markeredgecolor='white',
                    zorder=4, label=region)

        # connect dots with a faint line so bands read as a series
        valid = np.isfinite(betas)
        ax.plot(xpos[valid], betas[valid],
                color=color, lw=1.0, alpha=0.35, zorder=3)

        for xi, beta, se, p in zip(xpos, betas, ses, ps):
            star = sig_star(p)
            if star and np.isfinite(beta) and np.isfinite(se):
                tip = beta + se if beta >= 0 else beta - se
                _annotate_bar(ax, xi, tip, star, color)
            # track extremes including error bar tips
            if np.isfinite(beta) and np.isfinite(se):
                y_extremes.append(beta + se)
                y_extremes.append(beta - se)

    ax.axhline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
    ax.set_xticks(x_centers)
    ax.set_xticklabels([b.capitalize() for b in BANDS], fontsize=TICK_FS)
    ax.set_ylabel('β', fontsize=LABEL_FS)
    ax.set_title(
        f'Model 1 (bare)  |  {OUTCOME_LABELS[outcome]}',
        fontsize=TITLE_FS, fontweight='bold', pad=6)
    _style_ax(ax)

    # y-limits: span of beta±SE plus 30% headroom for stars
    if y_extremes:
        lo, hi = min(y_extremes), max(y_extremes)
        pad = (hi - lo) * 0.30 + 0.005
        ax.set_ylim(lo - pad, hi + pad)


def draw_model2_panel(ax, df: pd.DataFrame,
                      predictor: str, outcome: str):
    """
    Model 2 (controlled): mean ± SE errorbar plot.
    Predictor of interest: solid markers (LHP dark blue, RHP dark red).
    Covariates: open/distinct markers, covariate colours, offset slightly.
    All terms plotted at each band x-position with small horizontal offsets.
    """
    covariates = [CROSS_COVARIATE[predictor], 'output_lag']
    all_terms  = [predictor] + covariates

    # Check if semantic_sim exists
    check = df[
        (df['predictor_of_interest'] == predictor) &
        (df['outcome']               == outcome) &
        (df['model']                 == 'Model2') &
        (df['col']                   == 'semantic_sim')
    ]
    if not check.empty:
        all_terms.append('semantic_sim')

    x_centers = np.arange(len(BANDS), dtype=float)

    # Assign horizontal offsets: spread term×region combos symmetrically
    n_series  = len(all_terms) * len(REGIONS)
    slot_w    = 0.13
    offsets_all = np.linspace(-(n_series - 1) / 2 * slot_w,
                               (n_series - 1) / 2 * slot_w,
                               n_series)

    # Marker styles: predictor = filled circle; covariates = distinct open shapes
    markers = {predictor: 'o'}
    cov_markers = ['s', '^', 'D', 'v']
    for i, cv in enumerate(all_terms[1:]):
        markers[cv] = cov_markers[i % len(cov_markers)]

    colors_map = {'LHP': COLOR_LHP, 'RHP': COLOR_RHP}

    y_extremes = []
    slot_idx   = 0

    for term in all_terms:
        for region in REGIONS:
            betas, ses, ps = [], [], []
            for band in BANDS:
                b, s, p = get_beta(df, region, band, outcome,
                                   predictor, 'Model2', term)
                betas.append(b); ses.append(s); ps.append(p)

            betas = np.array(betas, dtype=float)
            ses   = np.array(ses,   dtype=float)
            xpos  = x_centers + offsets_all[slot_idx]

            if term == predictor:
                color     = colors_map[region]
                alpha     = 0.90
                msize     = 8
                lw        = 1.8
                mew       = 1.2
                mec       = 'white'
                zord      = 5
                linealpha = 0.40
            else:
                color     = COVARIATE_COLORS.get(term, COLOR_CROSS)
                alpha     = 0.65
                msize     = 6
                lw        = 1.2
                mew       = 1.5
                mec       = color
                zord      = 3
                linealpha = 0.25

            ax.errorbar(xpos, betas,
                        yerr=np.where(np.isfinite(ses), ses, 0),
                        fmt=markers[term],
                        color=color, alpha=alpha,
                        ecolor=color, elinewidth=lw,
                        capsize=4, capthick=lw,
                        markersize=msize,
                        markeredgewidth=mew,
                        markeredgecolor=mec,
                        zorder=zord)

            valid = np.isfinite(betas)
            ax.plot(xpos[valid], betas[valid],
                    color=color, lw=0.8, alpha=linealpha, zorder=zord - 1)

            # stars only on predictor of interest
            if term == predictor:
                for xi, beta, se, p in zip(xpos, betas, ses, ps):
                    star = sig_star(p)
                    if star and np.isfinite(beta) and np.isfinite(se):
                        tip = beta + se if beta >= 0 else beta - se
                        _annotate_bar(ax, xi, tip, star, color)

            for beta, se in zip(betas, ses):
                if np.isfinite(beta) and np.isfinite(se):
                    y_extremes.append(beta + se)
                    y_extremes.append(beta - se)

            slot_idx += 1

    ax.axhline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
    ax.set_xticks(x_centers)
    ax.set_xticklabels([b.capitalize() for b in BANDS], fontsize=TICK_FS)
    ax.set_ylabel('β', fontsize=LABEL_FS)
    ax.set_title(
        f'Model 2 (controlled)  |  {OUTCOME_LABELS[outcome]}',
        fontsize=TITLE_FS, fontweight='bold', pad=6)
    _style_ax(ax)

    if y_extremes:
        lo, hi = min(y_extremes), max(y_extremes)
        pad = (hi - lo) * 0.30 + 0.005
        ax.set_ylim(lo - pad, hi + pad)


# ============================================================================
# LEGEND BUILDER
# ============================================================================

def build_legend_handles(predictor: str, has_semsim: bool = False):
    cov_markers = ['s', '^', 'D']
    cross_term  = CROSS_COVARIATE[predictor]
    handles = [
        Line2D([0], [0], marker='o', color=COLOR_LHP, alpha=0.90,
               markersize=9, markeredgecolor='white', markeredgewidth=1.2,
               linewidth=0, label='LHP  (predictor)'),
        Line2D([0], [0], marker='o', color=COLOR_RHP, alpha=0.90,
               markersize=9, markeredgecolor='white', markeredgewidth=1.2,
               linewidth=0, label='RHP  (predictor)'),
        Line2D([0], [0], marker=cov_markers[0],
               color=COVARIATE_COLORS[cross_term], alpha=0.65,
               markersize=7, markeredgecolor=COVARIATE_COLORS[cross_term],
               markeredgewidth=1.5, linewidth=0,
               label=CROSS_LABELS[cross_term]),
        Line2D([0], [0], marker=cov_markers[1],
               color=COLOR_OUTLG, alpha=0.65,
               markersize=7, markeredgecolor=COLOR_OUTLG,
               markeredgewidth=1.5, linewidth=0,
               label='Output lag (ctrl)'),
    ]
    if has_semsim:
        handles.append(
            Line2D([0], [0], marker=cov_markers[2],
                   color=COLOR_SEMSIM, alpha=0.65,
                   markersize=7, markeredgecolor=COLOR_SEMSIM,
                   markeredgewidth=1.5, linewidth=0,
                   label='Semantic sim (ctrl)'))
    handles.append(
        Line2D([0], [0], color='none',
               label='* p<.05  ** p<.01  *** p<.001  (uncorrected)'))
    return handles


# ============================================================================
# FIGURE BUILDER
# ============================================================================

def make_predictor_figure(df: pd.DataFrame, predictor: str):
    """
    One figure for one predictor (T_lag or SP_lag).

    Layout:
      Rows : Model 1 (top) | Model 2 (bottom)
      Cols : Encoding (left) | Retrieval (right)
    """
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.patch.set_facecolor(BG_COLOR)

    for col_idx, outcome in enumerate(OUTCOMES):        # enc=0, ret=1
        draw_model1_panel(axes[0, col_idx], df, predictor, outcome)
        draw_model2_panel(axes[1, col_idx], df, predictor, outcome)

    # Row labels on the left
    for row_idx, model_lbl in enumerate(['Model 1\n(bare)', 'Model 2\n(controlled)']):
        axes[row_idx, 0].set_ylabel(
            f'β\n\n{model_lbl}', fontsize=LABEL_FS, fontweight='bold',
            labelpad=14)

    # Check if semantic_sim present
    has_semsim = not df[
        (df['predictor_of_interest'] == predictor) &
        (df['model'] == 'Model2') &
        (df['col']   == 'semantic_sim')
    ].empty

    legend_handles = build_legend_handles(predictor, has_semsim)
    fig.legend(handles=legend_handles,
               loc='lower center', ncol=len(legend_handles),
               fontsize=LEGEND_FS, frameon=True,
               facecolor='white', edgecolor=SPINE_COLOR,
               bbox_to_anchor=(0.5, -0.03))

    fig.suptitle(
        f'{PRED_LABELS[predictor]}  →  RSA  |  LHP vs RHP  |  {EXPERIMENT}',
        fontsize=SUPTITLE_FS, fontweight='bold',
        color=TEXT_COLOR, y=1.01)

    fig.tight_layout(rect=[0, 0.05, 1, 1])

    save_path = OUTPUT_DIR / f"summary_{predictor}_{EXPERIMENT}.png"
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    print(f"✓ Saved → {save_path.name}")
    plt.show()
    plt.close(fig)


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*70}")
    print(f"RSA Lag LMM Plot  |  {EXPERIMENT}")
    print(f"{'='*70}\n")

    df = load_master(EXPERIMENT)
    if df.empty:
        print("No data — exiting.")
        return

    for predictor in PREDICTORS:
        print(f"\n── Plotting {predictor} ──")
        make_predictor_figure(df, predictor)

    print(f"\n{'='*70}")
    print(f"✓ All plots saved to {OUTPUT_DIR}")
    print(f"{'='*70}")


if __name__ == '__main__':
    main()


RSA Lag LMM Plot  |  DBOY1

Loaded 160 rows for experiment 'DBOY1'

── Plotting T_lag ──
✓ Saved → summary_T_lag_DBOY1.png

── Plotting SP_lag ──
✓ Saved → summary_SP_lag_DBOY1.png

✓ All plots saved to rsa_lag_hc_bands_lmm_frac/plots


In [5]:
#!/usr/bin/env python3
"""
RSA Lag — Sequential LMM Analysis (4 Steps) — RAW SCALE, DBOY1 only
========================================================================
*** MIXED POWER variant (frac + osc) — reads from rsa_lag_hc_bands_mixed/ ***

Representational vectors were built as:
    mixed[freq, ch] = ret_frac_f{freq} + ret_osc_f{freq}
then sliced to band frequency indices (freq × channel, flattened).

Betas are in original units:
  RSA_r         : Pearson r
  T_lag         : number of list positions separating the two items
  SP_lag        : Euclidean distance in store-coordinate units
  semantic_sim  : Word2Vec cosine similarity between recalled words
  output_lag    : number of recall output positions between the two items

Predictors tested in all steps: T_lag, SP_lag, semantic_sim

Steps
-----
  Step 1 — RSA_r ~ predictor + output_lag + (1|subj)
            All bands pooled, both hemispheres.
            Run separately for each of: T_lag, SP_lag, semantic_sim.

  Step 2 — RSA_r ~ predictor * band (explicit dummies, theta = reference)
                 + output_lag + (1|subj)
            Tests whether predictor slope differs across bands.
            Run separately for each predictor.

  Step 3 — RSA_r ~ predictor * hemisphere + output_lag + (1|subj)
            Theta band only.  Tests LHP vs RHP asymmetry.
            Also run separately for LHP and RHP.
            Run for each predictor.

  Step 4 — RSA_r ~ T_lag + SP_lag + semantic_sim + output_lag + (1|subj)
            Theta band only.  Tests independence of all three predictors
            simultaneously.

Input  : ./rsa_lag_hc_bands_mixed/ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag_mixed.csv
Output : ./rsa_lag_hc_bands_mixed/SEQUENTIAL_LMM_RAW_DBOY1_mixed_results.csv
         ./rsa_lag_hc_bands_mixed/SEQUENTIAL_LMM_RAW_DBOY1_mixed_results.txt
"""

import warnings
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

INPUT_DIR  = Path('./rsa_lag_hc_bands_mixed')
OUTPUT_DIR = Path('./rsa_lag_hc_bands_mixed')

BANDS         = ['theta', 'alpha', 'beta', 'gamma']
NON_REF_BANDS = ['alpha', 'beta', 'gamma']   # theta = reference

# All three predictors treated symmetrically across Steps 1–3
PREDICTORS = ['T_lag', 'SP_lag', 'semantic_sim']

OUTCOME = 'RSA_r'


# ============================================================================
# DATA LOADING
# ============================================================================

def load_data() -> Optional[pd.DataFrame]:
    fpath = INPUT_DIR / "ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag_mixed.csv"
    if not fpath.exists():
        print(f"  ✗ File not found: {fpath}")
        return None

    df = pd.read_csv(fpath)
    print(f"  Loaded {fpath.name}  ({len(df):,} rows)")

    if 'hemisphere' not in df.columns:
        df['hemisphere'] = df['region'].map({'LHP': 'left', 'RHP': 'right'})

    df['hemisphere_dummy'] = (df['hemisphere'] == 'right').astype(int)

    # Report semantic_sim coverage
    n_total  = len(df)
    n_semsim = df['semantic_sim'].notna().sum()
    print(f"\n  Total rows       : {n_total:,}")
    print(f"  Subjects         : {df['subject'].nunique()}")
    print(f"  Sessions         : {df['session'].nunique()}")
    print(f"  Bands            : {sorted(df['band'].unique().tolist())}")
    print(f"  Regions          : {sorted(df['region'].unique().tolist())}")
    print(f"  semantic_sim     : {n_semsim:,} non-NaN "
          f"({100*n_semsim/n_total:.1f}% coverage)")

    print(f"\n  Predictor scales (raw):")
    for col in ['RSA_r', 'T_lag', 'SP_lag', 'semantic_sim', 'output_lag']:
        if col in df.columns:
            s = df[col].dropna()
            print(f"    {col:<14} mean={s.mean():.3f}  "
                  f"sd={s.std():.3f}  "
                  f"range=[{s.min():.3f}, {s.max():.3f}]  "
                  f"N={len(s):,}")

    return df


# ============================================================================
# LMM FITTING
# ============================================================================

def fit_lmm(df: pd.DataFrame,
            formula_cols: List[str],
            formula_rhs: str,
            label: str) -> Optional[object]:
    """
    Fit RSA_r ~ formula_rhs.

    Random effects strategy (tried in order):
      1. Nested: (1|subject) + (1|subject:session)  via vc_formula
      2. Simple : (1|subject) only — fallback if nested hits a boundary
    """
    df = df.copy()
    df['subj_sess'] = df['subject'].astype(str) + '_' + df['session'].astype(str)

    keep = [OUTCOME] + [c for c in formula_cols if c in df.columns] \
           + ['subject', 'subj_sess']
    df   = df[keep].dropna()

    n = len(df)
    if n < 20:
        print(f"  [{label}] Too few rows ({n}) — skipping")
        return None

    formula = f"{OUTCOME} ~ {formula_rhs}"
    print(f"\n  [{label}] {formula}")
    print(f"  [{label}] N={n:,}  subjects={df['subject'].nunique()}")

    def _is_boundary(res) -> bool:
        if res is None or not np.isfinite(res.llf):
            return True
        for k in res.params.index:
            if 'subj_sess' in k.lower():
                if abs(res.params[k] - 16.0) < 1e-6:
                    return True
                if not np.isfinite(res.bse[k]):
                    return True
        return False

    # Attempt 1: nested random effects
    result = None
    for method in ['lbfgs', 'nm', 'powell']:
        try:
            model = MixedLM.from_formula(
                formula,
                data       = df,
                groups     = df['subject'],
                vc_formula = {'subj_sess': '0 + C(subj_sess)'},
            )
            res = model.fit(reml=True, method=method)
            if np.isfinite(res.llf) and not _is_boundary(res):
                print(f"  [{label}] nested RE  optimizer={method}  "
                      f"converged={getattr(res, 'converged', None)}")
                result = res
                break
        except Exception:
            pass

    # Attempt 2: subject-only random effects
    if result is None or _is_boundary(result):
        print(f"  [{label}] ⚠ Nested RE boundary — "
              f"falling back to (1|subject) only")
        for method in ['lbfgs', 'nm', 'powell']:
            try:
                model = MixedLM.from_formula(
                    formula,
                    data   = df,
                    groups = df['subject'],
                )
                res = model.fit(reml=True, method=method)
                if np.isfinite(res.llf):
                    print(f"  [{label}] subject-only RE  optimizer={method}  "
                          f"converged={getattr(res, 'converged', None)}")
                    result = res
                    break
            except Exception as e:
                print(f"  [{label}] subject-only {method} failed: {e}")
                result = None

    if result is None:
        print(f"  [{label}] WARNING: all fits unsuccessful")
    return result


# ============================================================================
# RESULT FORMATTING
# ============================================================================

def sig_stars(p: float) -> str:
    return ('***' if p < 0.001 else '**' if p < 0.01 else
            '*'   if p < 0.05  else '†'  if p < 0.10  else '')


def result_to_df(result, step: str, model_desc: str,
                 hemisphere: str = 'combined',
                 predictor: str = '') -> pd.DataFrame:
    if result is None:
        return pd.DataFrame()

    rows = []
    for term in result.params.index:
        p_val = result.pvalues[term]
        is_re = (not np.isfinite(result.bse[term]) or
                 not np.isfinite(p_val) or
                 'var' in term.lower())
        rows.append({
            'step':       step,
            'predictor':  predictor,
            'model':      model_desc,
            'hemisphere': hemisphere,
            'term':       term,
            'beta':       result.params[term],
            'se':         result.bse[term],
            'z':          result.tvalues[term],
            'p_raw':      p_val,
            'p_fdr':      np.nan,
            'is_re':      is_re,
            'nobs':       int(result.nobs),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    fe_mask = (~df['is_re']) & df['p_raw'].notna() & np.isfinite(df['p_raw'])
    if fe_mask.sum() > 0:
        _, fdr_vals = fdrcorrection(df.loc[fe_mask, 'p_raw'].values)
        df.loc[fe_mask, 'p_fdr'] = fdr_vals

    return df


def print_table(df: pd.DataFrame, title: str):
    if df.empty:
        print(f"\n  [no results for: {title}]")
        return

    sep  = '=' * 95
    sep2 = '-' * 95
    hdr  = (f"{'Term':<45} {'β':>10} {'SE':>10} {'z':>8} "
            f"{'p_raw':>10} {'p_fdr':>10} {'N':>8}")

    print(f"\n{sep}")
    print(title)
    print(sep2)
    print(hdr)
    print(sep2)

    fe = df[~df['is_re']] if 'is_re' in df.columns else df
    for _, row in fe.iterrows():
        p_fdr = row.get('p_fdr', np.nan)
        p_fdr_str = f"{p_fdr:>10.4f}" if np.isfinite(p_fdr) else f"{'—':>10}"
        stars = sig_stars(p_fdr) if np.isfinite(p_fdr) else ''
        print(f"  {row['term']:<43} {row['beta']:>10.5f} {row['se']:>10.5f} "
              f"{row['z']:>8.3f} {row['p_raw']:>10.4f} "
              f"{p_fdr_str} {int(row['nobs']):>8,}  {stars}")

    re = df[df['is_re']] if 'is_re' in df.columns else pd.DataFrame()
    if not re.empty:
        print(f"  {'— random effects —'}")
        for _, row in re.iterrows():
            beta_str = f"{row['beta']:>10.5f}" if np.isfinite(row['beta']) \
                       else f"{'—':>10}"
            print(f"  {row['term']:<43} {beta_str}")

    print(sep2)
    print("  Outcome = RSA_r (raw Pearson r, mixed power = frac+osc).  DBOY1 only.")
    print("  FDR: BH within fixed effects only.  "
          "† p<.10  * p<.05  ** p<.01  *** p<.001")
    print(f"  Reference: band=theta, hemisphere=LHP")
    print(sep)


# ============================================================================
# STEP 1 — univariate effects (one predictor at a time)
# ============================================================================

def step1(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 1 — Does RSA_r track each predictor?")
    print(f"{'#'*70}")
    print("  Data: all bands pooled, both hemispheres")
    print(f"  Predictors: {PREDICTORS}")

    results = []
    for pred in PREDICTORS:
        res = fit_lmm(
            df,
            formula_cols = [pred, 'output_lag'],
            formula_rhs  = f'{pred} + output_lag',
            label        = f'Step1 {pred}',
        )
        rdf = result_to_df(res, step='Step1',
                           model_desc=f'RSA_r ~ {pred} + output_lag',
                           predictor=pred)
        print_table(rdf,
                    f"Step 1 | RSA_r ~ {pred} + output_lag  [all bands, combined]")
        results.append(rdf)

    return results


# ============================================================================
# STEP 2 — band specificity (predictor × band interaction)
# ============================================================================

def step2(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 2 — Is each effect theta-specific?")
    print(f"{'#'*70}")
    print("  Data: all bands pooled, both hemispheres")
    print("  Reference band: theta")

    df = df.copy()
    for band in NON_REF_BANDS:
        df[f'band_{band}'] = (df['band'] == band).astype(float)

    results = []
    for pred in PREDICTORS:

        for band in NON_REF_BANDS:
            df[f'{pred}_x_{band}'] = df[pred] * df[f'band_{band}']

        band_cols  = [f'band_{b}'     for b in NON_REF_BANDS]
        inter_cols = [f'{pred}_x_{b}' for b in NON_REF_BANDS]

        formula_cols = [pred] + band_cols + inter_cols + ['output_lag']
        formula_rhs  = (f'{pred} + '
                        + ' + '.join(band_cols) + ' + '
                        + ' + '.join(inter_cols)
                        + ' + output_lag')

        res = fit_lmm(df, formula_cols, formula_rhs, label=f'Step2 {pred}')
        rdf = result_to_df(res, step='Step2',
                           model_desc=f'RSA_r ~ {pred} * band + output_lag',
                           predictor=pred)
        print_table(rdf,
                    f"Step 2 | RSA_r ~ {pred} * band + output_lag  "
                    f"[all bands, combined]\n"
                    f"  KEY: {pred}_x_{{band}} significant → slope ≠ theta")
        results.append(rdf)

    return results


# ============================================================================
# STEP 3 — hemispheric asymmetry (theta band only)
# ============================================================================

def step3(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 3 — Is each theta effect lateralized?  (theta band only)")
    print(f"{'#'*70}")

    theta_df = df[df['band'] == 'theta'].copy()
    print(f"  Theta rows: {len(theta_df):,}  "
          f"subjects: {theta_df['subject'].nunique()}")

    results = []
    for pred in PREDICTORS:
        inter_col = f'{pred}_x_hemi'
        theta_df[inter_col] = theta_df[pred] * theta_df['hemisphere_dummy']

        # Combined: interaction test
        res = fit_lmm(
            theta_df,
            formula_cols = [pred, 'hemisphere_dummy', inter_col, 'output_lag'],
            formula_rhs  = (f'{pred} + hemisphere_dummy + {inter_col}'
                            f' + output_lag'),
            label        = f'Step3 {pred} combined',
        )
        rdf = result_to_df(res, step='Step3',
                           model_desc=f'RSA_r ~ {pred} * hemisphere + output_lag',
                           hemisphere='combined',
                           predictor=pred)
        print_table(rdf,
                    f"Step 3 | RSA_r ~ {pred} * hemisphere + output_lag  "
                    f"[theta, combined]\n"
                    f"  KEY TERM: {inter_col}  "
                    f"(significant → LHP slope ≠ RHP slope)")
        results.append(rdf)

        # Simple slopes per hemisphere
        for hemi, hemi_label in [('left', 'LHP'), ('right', 'RHP')]:
            hd = theta_df[theta_df['hemisphere'] == hemi].copy()
            res_h = fit_lmm(
                hd,
                formula_cols = [pred, 'output_lag'],
                formula_rhs  = f'{pred} + output_lag',
                label        = f'Step3 {pred} {hemi_label}',
            )
            rdf_h = result_to_df(res_h, step='Step3',
                                 model_desc=f'RSA_r ~ {pred} + output_lag',
                                 hemisphere=hemi_label,
                                 predictor=pred)
            print_table(rdf_h,
                        f"Step 3 | RSA_r ~ {pred} + output_lag  "
                        f"[theta, {hemi_label} only]")
            results.append(rdf_h)

    return results


# ============================================================================
# STEP 4 — mutual control of all three predictors (theta only)
# ============================================================================

def step4(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 4 — Do T_lag, SP_lag, and semantic_sim survive mutual control?")
    print("         (theta band only)")
    print(f"{'#'*70}")

    theta_df = df[df['band'] == 'theta'].copy()
    print(f"  Theta rows (before semsim dropna): {len(theta_df):,}  "
          f"subjects: {theta_df['subject'].nunique()}")

    # Report complete-case count
    complete = theta_df[['T_lag', 'SP_lag', 'semantic_sim',
                          'output_lag', OUTCOME]].dropna()
    print(f"  Theta rows (complete cases):        {len(complete):,}")

    formula_cols = ['T_lag', 'SP_lag', 'semantic_sim', 'output_lag']
    formula_rhs  = 'T_lag + SP_lag + semantic_sim + output_lag'

    results = []
    for subset_label, subset_df in [
            ('combined', theta_df),
            ('LHP',      theta_df[theta_df['hemisphere'] == 'left']),
            ('RHP',      theta_df[theta_df['hemisphere'] == 'right'])]:

        res = fit_lmm(
            subset_df,
            formula_cols = formula_cols,
            formula_rhs  = formula_rhs,
            label        = f'Step4 {subset_label}',
        )
        rdf = result_to_df(
            res, step='Step4',
            model_desc='RSA_r ~ T_lag + SP_lag + semantic_sim + output_lag',
            hemisphere=subset_label,
            predictor='all',
        )
        print_table(rdf,
                    f"Step 4 | RSA_r ~ T_lag + SP_lag + semantic_sim + output_lag"
                    f"  [theta, {subset_label}]")
        results.append(rdf)

    return results


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*70}")
    print("RSA LAG — SEQUENTIAL LMM  |  DBOY1 only  |  RAW SCALE  |  MIXED (frac+osc)")
    print(f"Predictors: T_lag, SP_lag, semantic_sim  (all treated in parallel)")
    print(f"{'='*70}")

    df = load_data()
    if df is None or df.empty:
        print("No data loaded. Exiting.")
        return

    all_results = []
    all_results += step1(df)
    all_results += step2(df)
    all_results += step3(df)
    all_results += step4(df)

    # Save CSV
    if all_results:
        master = pd.concat(all_results, ignore_index=True)
        csv_path = OUTPUT_DIR / 'SEQUENTIAL_LMM_RAW_DBOY1_mixed_results.csv'
        master.to_csv(csv_path, index=False)
        print(f"\n  ✓ Results → {csv_path}  ({len(master):,} rows)")

        # Save text
        txt_path = OUTPUT_DIR / 'SEQUENTIAL_LMM_RAW_DBOY1_mixed_results.txt'
        lines = []
        for rdf in all_results:
            if rdf.empty:
                continue
            lines.append(f"\n{'='*80}")
            lines.append(f"{rdf['step'].iloc[0]}  |  "
                         f"pred={rdf['predictor'].iloc[0]}  |  "
                         f"{rdf['model'].iloc[0]}  |  "
                         f"{rdf['hemisphere'].iloc[0]}")
            lines.append(f"{'='*80}")
            for _, row in rdf[~rdf['is_re']].iterrows():
                p_fdr = row['p_fdr']
                stars = sig_stars(p_fdr) if np.isfinite(p_fdr) else ''
                p_fdr_s = f"{p_fdr:.4f}" if np.isfinite(p_fdr) else '—'
                lines.append(
                    f"  {row['term']:<40} β={row['beta']:>10.5f}  "
                    f"SE={row['se']:>9.5f}  z={row['z']:>7.3f}  "
                    f"p={row['p_raw']:>8.4f}  p_fdr={p_fdr_s:>8}  {stars}"
                )
        with open(txt_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"  ✓ Text    → {txt_path}")

    print(f"\n{'='*70}")
    print("INTERPRETATION GUIDE")
    print(f"{'='*70}")
    print("""
  Step 1: Each predictor tested separately controlling for output_lag.
          T_lag/SP_lag significant → genuine spatial/temporal clustering.
          semantic_sim significant → neural similarity tracks word meaning.

  Step 2: predictor_x_alpha/beta/gamma significant?
          → theta slope differs from that band → theta specificity confirmed.
          NOT significant → effect is broadband.

  Step 3: predictor_x_hemi significant?
          → LHP and RHP slopes differ.
          Simple slopes reveal which hemisphere drives the effect.

  Step 4: All three predictors (T_lag, SP_lag, semantic_sim) + output_lag
          entered simultaneously in theta band.
          Surviving effects are independent of the other two dimensions.
          Note: rows with NaN semantic_sim are dropped from Step 4 models.

  NOTE: RSA vectors used mixed power (frac + osc per freq × channel).
    """)


if __name__ == '__main__':
    main()


RSA LAG — SEQUENTIAL LMM  |  DBOY1 only  |  RAW SCALE  |  MIXED (frac+osc)
Predictors: T_lag, SP_lag, semantic_sim  (all treated in parallel)
  Loaded ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag_mixed.csv  (13,028 rows)

  Total rows       : 13,028
  Subjects         : 34
  Sessions         : 6
  Bands            : ['alpha', 'beta', 'gamma', 'theta']
  Regions          : ['LHP', 'RHP']
  semantic_sim     : 9,960 non-NaN (76.5% coverage)

  Predictor scales (raw):
    RSA_r          mean=0.195  sd=0.399  range=[-1.000, 1.000]  N=13,028
    T_lag          mean=4.523  sd=2.914  range=[1.000, 11.000]  N=13,028
    SP_lag         mean=70.149  sd=30.696  range=[12.962, 142.722]  N=13,028
    semantic_sim   mean=0.278  sd=0.176  range=[-0.059, 0.821]  N=9,960
    output_lag     mean=2.450  sd=1.560  range=[1.000, 10.000]  N=13,028

######################################################################
STEP 1 — Does RSA_r track each predictor?
######################################################